# DenseCL Self-Supervised Pretraining — Kaggle Training Notebook

Runs the exact same DenseCL dual-loss (global InfoNCE + dense
foreground-masked correspondence) SSL pretraining pipeline already
developed and smoke-tested locally (RTX 3050 4GB, 5-epoch run), on a
Kaggle GPU session (T4 x2, 16GB each — only **one** GPU is used; this
pipeline does not implement multi-GPU training).

**How this notebook is organized:**
1. One single editable cell (`kaggle_config.py`) holds every input path
   and training hyperparameter — edit that cell only, nothing else needs
   touching for a normal re-run.
2. A few setup cells install/check dependencies.
3. One `%%writefile` cell per source module, writing the exact same code
   already verified locally into a flat `/kaggle/working/densecl_src/`
   directory on the Kaggle instance.
4. A cell that writes the already-frozen, already-used writer-level
   test/validation split JSONs (identical content to the ones used for the
   local run — not regenerated, just copied in, so held-out writers never
   change).
5. The training run itself, then a small results-verification cell.

**Before running:** upload your signature dataset as a Kaggle Dataset (see
the note above the `kaggle_config.py` cell for exactly what it must
contain), attach it to this notebook as an input, then edit `DATA_ROOT` in
the config cell to match its mounted path. Then Commit & Run All.

## 1. Setup

In [ ]:
# Kaggle's base image already ships torch + CUDA and (usually) opencv, but
# this makes the notebook self-contained even on a fresh/changed image.
!pip install -q opencv-python-headless pandas tqdm


In [ ]:
import torch

print(f"Torch version : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count     : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

!nvidia-smi


## 2. Configuration — the ONLY cell you should need to edit

**Dataset upload requirement:** create a Kaggle Dataset whose contents,
once mounted, directly contain these four folders (matching the local
project's `DenseCL_approach/data/all/` layout exactly — one subfolder per
writer inside each):

```
<your-dataset-root>/
  BHSig260_Bengali/<writer_id>/<file>
  BHSig260_Hindi/<writer_id>/<file>
  CEDAR/<writer_id>/<file>
  Mendeley/<writer_id>/<file>
```

After attaching it to this notebook, run `!ls /kaggle/input/` to find its
actual mounted folder name, then set `DATA_ROOT` below to that path. See
the chat / `progress_so_far.md` for exactly which raw files (no
preprocessing needed — this notebook preprocesses on the fly, same as the
local pipeline) go in each folder.

`RUN_NAME` must stay `"densecl_pretrain_v1"` if you want this run to be
directly comparable to the existing 5-epoch local smoke-test run recorded
in `progress_so_far.md` / `analysis_to_do.md` — change it only if you
intend to start a distinctly-named new run.

In [ ]:
import os

os.makedirs("/kaggle/working/densecl_src", exist_ok=True)


In [ ]:
%%writefile /kaggle/working/densecl_src/kaggle_config.py
from __future__ import annotations

from pathlib import Path

# ============================================================================
# EDIT THIS SECTION — every input path and training knob lives here.
# ============================================================================

# --- Paths -----------------------------------------------------------------
# Must point at the folder that DIRECTLY contains BHSig260_Bengali/,
# BHSig260_Hindi/, CEDAR/, Mendeley/ — run `!ls /kaggle/input/` after
# attaching your dataset to find the exact mounted name.
DATA_ROOT = Path("/kaggle/input/densecl-signature-data/all")

# Everything this run produces (loss CSVs, checkpoints) lands here.
# /kaggle/working/ is what "Commit and Run All" persists as the notebook's
# output, so nothing extra needs to be zipped/saved manually.
RESULTS_DIR = Path("/kaggle/working/results/training")

# --- Run identity ------------------------------------------------------------
RUN_NAME = "densecl_pretrain_v1"  # keep this to match the local 5-epoch run

# --- Schedule ----------------------------------------------------------------
# Matches the local smoke-test run exactly (5 epochs, 1 warmup epoch). The
# "real" budget the local run's own TrainConfig comments call out is 50
# epochs / 5 warmup epochs — bump these once a short run here looks healthy.
NUM_EPOCHS = 5
WARMUP_EPOCHS = 1
SAVE_FREQUENCY = 5  # epochs between checkpoint saves (also always saves the last epoch)

# --- Batch / dataloader --------------------------------------------------
# batch_size=8 matches the local run (empirically tuned for a 4GB GPU).
# Kaggle's T4 has 16GB — you can raise this a lot; the learning rate below
# scales automatically with batch size (see BASE_LR/BASE_LR_BATCH_SIZE), so
# you do not need to hand-tune LR if you change this.
BATCH_SIZE = 8
NUM_WORKERS = 2  # 0 was used locally only because Windows multiprocessing is fragile; Kaggle (Linux) handles workers fine

# --- Optimizer (SGD + linear LR-scaling rule, MoCo-standard base_lr=0.03 @ batch=256) ---
BASE_LR = 0.03
BASE_LR_BATCH_SIZE = 256
SGD_MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4

# --- DenseCL-specific ------------------------------------------------------
ENCODER_MOMENTUM = 0.999   # EMA momentum for the momentum encoder/heads
QUEUE_SIZE = 4096          # global-branch negative queue size
TEMPERATURE_GLOBAL = 0.2
TEMPERATURE_DENSE = 0.2
LAMBDA_GLOBAL = 1.0
LAMBDA_DENSE = 1.0

# --- Reproducibility ---------------------------------------------------------
SEED = 42

# ============================================================================
# End of editable section.
# ============================================================================


In [ ]:
import sys

sys.path.insert(0, "/kaggle/working/densecl_src")


## 3. Held-out writer splits (test + SSL validation)

These are the exact same writer-level splits already used for the local
training run (`DenseCL_approach/data/{test_set_writer_split,
validation_set_writer_split}/*.json`) — written here verbatim, not
regenerated, so the held-out writers can never silently drift between the
local run and this one.

In [ ]:
import json
import os

TEST_SPLITS = {
    "CEDAR": {
        "dataset": "CEDAR",
        "total_writers": 55,
        "num_test_writers": 15,
        "seed": 42,
        "test_writer_ids": [
            "10",
            "11",
            "14",
            "15",
            "16",
            "17",
            "22",
            "23",
            "25",
            "34",
            "40",
            "43",
            "46",
            "49",
            "52"
        ]
    },
    "BHSig260_Bengali": {
        "dataset": "BHSig260_Bengali",
        "total_writers": 100,
        "num_test_writers": 30,
        "seed": 42,
        "test_writer_ids": [
            "1",
            "11",
            "12",
            "19",
            "20",
            "21",
            "24",
            "31",
            "33",
            "34",
            "35",
            "37",
            "40",
            "57",
            "58",
            "60",
            "67",
            "71",
            "73",
            "77",
            "79",
            "82",
            "85",
            "87",
            "89",
            "9",
            "93",
            "94",
            "95",
            "97"
        ]
    },
    "BHSig260_Hindi": {
        "dataset": "BHSig260_Hindi",
        "total_writers": 160,
        "num_test_writers": 30,
        "seed": 42,
        "test_writer_ids": [
            "10",
            "104",
            "105",
            "106",
            "119",
            "12",
            "122",
            "124",
            "130",
            "134",
            "135",
            "144",
            "149",
            "15",
            "150",
            "152",
            "155",
            "18",
            "19",
            "33",
            "51",
            "52",
            "58",
            "71",
            "79",
            "80",
            "87",
            "90",
            "92",
            "98"
        ]
    }
}

VALIDATION_SPLITS = {
    "CEDAR": {
        "dataset": "CEDAR",
        "total_writers": 55,
        "num_test_writers_excluded": 15,
        "num_eligible_writers": 40,
        "num_validation_writers": 5,
        "seed": 42,
        "validation_writer_ids": [
            "12",
            "21",
            "30",
            "31",
            "33"
        ]
    },
    "BHSig260_Bengali": {
        "dataset": "BHSig260_Bengali",
        "total_writers": 100,
        "num_test_writers_excluded": 30,
        "num_eligible_writers": 70,
        "num_validation_writers": 10,
        "seed": 42,
        "validation_writer_ids": [
            "14",
            "27",
            "28",
            "30",
            "46",
            "49",
            "52",
            "62",
            "66",
            "8"
        ]
    },
    "BHSig260_Hindi": {
        "dataset": "BHSig260_Hindi",
        "total_writers": 160,
        "num_test_writers_excluded": 30,
        "num_eligible_writers": 130,
        "num_validation_writers": 15,
        "seed": 42,
        "validation_writer_ids": [
            "108",
            "109",
            "11",
            "126",
            "127",
            "13",
            "132",
            "140",
            "21",
            "23",
            "25",
            "28",
            "36",
            "74",
            "99"
        ]
    },
    "Mendeley": {
        "dataset": "Mendeley",
        "total_writers": 200,
        "num_test_writers_excluded": 0,
        "num_eligible_writers": 200,
        "num_validation_writers": 20,
        "seed": 42,
        "validation_writer_ids": [
            "a_(104)",
            "a_(105)",
            "a_(106)",
            "a_(119)",
            "a_(12)",
            "a_(122)",
            "a_(124)",
            "a_(130)",
            "a_(149)",
            "a_(150)",
            "a_(152)",
            "a_(155)",
            "a_(162)",
            "a_(197)",
            "a_(44)",
            "a_(55)",
            "a_(66)",
            "a_(75)",
            "a_(89)",
            "a_(9)"
        ]
    }
}

test_split_dir = "/kaggle/working/densecl_src/data/test_set_writer_split"
validation_split_dir = "/kaggle/working/densecl_src/data/validation_set_writer_split"
os.makedirs(test_split_dir, exist_ok=True)
os.makedirs(validation_split_dir, exist_ok=True)

for dataset_name, split in TEST_SPLITS.items():
    out_path = os.path.join(test_split_dir, f"{dataset_name}_test_writers.json")
    with open(out_path, "w") as f:
        json.dump(split, f, indent=2)
    print(f"Wrote {out_path} ({split['num_test_writers']} test writers)")

for dataset_name, split in VALIDATION_SPLITS.items():
    out_path = os.path.join(validation_split_dir, f"{dataset_name}_validation_writers.json")
    with open(out_path, "w") as f:
        json.dump(split, f, indent=2)
    print(f"Wrote {out_path} ({split['num_validation_writers']} validation writers)")


## 4. Source modules

Each cell below writes one module, verbatim from the already-verified
local project code (only `test_set_creation.py`, `validation_set_creation.py`,
and `train.py` have the two path constants re-pointed at `kaggle_config.py`
— everything else, including all matching/masking/augmentation/model/loss
logic, is byte-identical to the local version).

In [ ]:
%%writefile /kaggle/working/densecl_src/preprocess.py
"""Deterministic preprocessing pipeline for offline signature images.

Pipeline: grayscale -> Otsu binarize -> crop to ink bounding box ->
square-pad -> resize to 256x256 -> Otsu again -> clean binary image.

The output is a 256x256 uint8 binary image with ink strokes at maximum
intensity (255) against a uniform black (0) background, matching the
preprocessing described for the reconstruction-SSL baseline (Fig 3.2 of
the thesis report) and reused unchanged for the DenseCL pretext.
"""

from __future__ import annotations

from pathlib import Path

import cv2
import numpy as np

TARGET_SIZE = 256


def otsu_binarize(gray: np.ndarray) -> np.ndarray:
    """Otsu threshold with inversion so ink pixels become the foreground (255).

    Public because `augment.py` reuses this exact step to re-binarize every
    augmented view after geometric/intensity perturbations, keeping the
    "always clean binary in, clean binary out" invariant throughout.
    """
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return binary


def _crop_to_ink_bbox(gray: np.ndarray, ink_mask: np.ndarray) -> np.ndarray:
    """Crop `gray` to the tight bounding box of the non-zero pixels in `ink_mask`."""
    ys, xs = np.nonzero(ink_mask)
    if ys.size == 0 or xs.size == 0:
        # No ink detected (blank/corrupt scan) - fall back to the full image
        # rather than crashing on an empty crop.
        return gray
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    return gray[y0:y1, x0:x1]


def _square_pad(gray: np.ndarray, fill_value: int = 255) -> np.ndarray:
    """Center `gray` on a square canvas, padding with `fill_value` (white)."""
    h, w = gray.shape
    side = max(h, w)
    top = (side - h) // 2
    bottom = side - h - top
    left = (side - w) // 2
    right = side - w - left
    return cv2.copyMakeBorder(
        gray, top, bottom, left, right,
        borderType=cv2.BORDER_CONSTANT, value=fill_value,
    )


def preprocess_signature(gray: np.ndarray, target_size: int = TARGET_SIZE) -> np.ndarray:
    """Run the full deterministic preprocessing pipeline on one signature image.

    Parameters
    ----------
    gray:
        Single-channel (grayscale) signature image, as returned by
        ``cv2.imread(path, cv2.IMREAD_GRAYSCALE)``.
    target_size:
        Side length of the final square output image.

    Returns
    -------
    np.ndarray
        A ``target_size x target_size`` uint8 binary image with ink
        strokes at maximum intensity (255) against a black (0) background.
    """
    if gray.ndim != 2:
        raise ValueError(f"Expected a single-channel grayscale image, got shape {gray.shape}")

    # Step 1: locate ink strokes via Otsu binarization. This mask is used
    # only to find the crop region and is discarded afterwards.
    ink_mask = otsu_binarize(gray)

    # Step 2: crop to the tight bounding box around the ink so writer- and
    # scanner-dependent margins don't affect downstream scale.
    cropped = _crop_to_ink_bbox(gray, ink_mask)

    # Step 3: pad to a square canvas (centered) to preserve aspect ratio
    # and avoid the anisotropic distortion a direct resize would introduce.
    squared = _square_pad(cropped, fill_value=255)

    # Step 4: resize to the fixed resolution the encoder consumes.
    resized = cv2.resize(
        squared, (target_size, target_size), interpolation=cv2.INTER_AREA
    )

    # Step 5: re-binarize to remove grayscale interpolation artifacts
    # introduced by resizing, yielding a clean binary signature.
    clean_binary = otsu_binarize(resized)

    return clean_binary


def load_and_preprocess(path: str | Path) -> tuple[np.ndarray, np.ndarray]:
    """Load an image from disk and return `(original_grayscale, preprocessed)`."""
    path = Path(path)
    original_gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if original_gray is None:
        raise FileNotFoundError(f"Could not read image at {path}")
    preprocessed = preprocess_signature(original_gray)
    return original_gray, preprocessed


if __name__ == "__main__":
    import sys

    if len(sys.argv) != 2:
        print("Usage: python preprocess.py <path-to-signature-image>")
        raise SystemExit(1)

    orig, clean = load_and_preprocess(sys.argv[1])
    print(f"Original shape:     {orig.shape}")
    print(f"Preprocessed shape: {clean.shape}, dtype={clean.dtype}, "
          f"unique values={np.unique(clean)}")


In [ ]:
%%writefile /kaggle/working/densecl_src/mask.py
"""Foreground (ink) mask computation for the encoder's dense feature grid.

The dense DenseCL correspondence loss operates on the encoder's 32x32
feature grid, not on raw pixels, so the pixel-level binary mask already
present in a preprocessed signature (ink=255, background=0) has to be
summarized down to one "how much ink is in this cell" number per grid
cell, and then turned into a foreground/background decision per cell.
"""

from __future__ import annotations

import numpy as np

GRID_SIZE = 32


def compute_ink_coverage(binary_image: np.ndarray, grid_size: int = GRID_SIZE) -> np.ndarray:
    """Fraction of ink pixels in each non-overlapping grid cell.

    Parameters
    ----------
    binary_image:
        A square binary image (e.g. the output of `preprocess_signature`),
        ink pixels > 0, background == 0.
    grid_size:
        Number of cells per side. Must evenly divide the image side length
        (32 for a 256x256 image, matching the encoder's three stride-2
        downsamples: 256 / 8 = 32).

    Returns
    -------
    np.ndarray
        `(grid_size, grid_size)` float32 array; each value is the fraction
        (0.0-1.0) of ink pixels inside that cell's pixel block.
    """
    h, w = binary_image.shape
    if h != w:
        raise ValueError(f"Expected a square image, got shape {binary_image.shape}")
    if h % grid_size != 0:
        raise ValueError(
            f"Image side length {h} is not evenly divisible by grid_size {grid_size}"
        )

    cell = h // grid_size
    ink = (binary_image > 0).astype(np.float32)
    coverage = ink.reshape(grid_size, cell, grid_size, cell).mean(axis=(1, 3))
    return coverage


def foreground_mask_from_coverage(coverage: np.ndarray, min_coverage: float = 0.0) -> np.ndarray:
    """Turn per-cell ink coverage into a foreground/background decision.

    A cell counts as foreground if its ink coverage is strictly greater
    than `min_coverage`. The default (0.0) means "any ink pixel in the
    cell counts" - the grid-cell equivalent of the pixel-level rule
    already used for the reconstruction pretext's foreground-weighted MSE
    (Eq. 3.5 of the thesis report: w(p) = w_fg if t(p) > 0 else w_bg).
    """
    return coverage > min_coverage


def compute_foreground_mask(
    binary_image: np.ndarray, grid_size: int = GRID_SIZE, min_coverage: float = 0.0
) -> tuple[np.ndarray, np.ndarray]:
    """Convenience wrapper: binary image -> (foreground_mask, ink_coverage)."""
    coverage = compute_ink_coverage(binary_image, grid_size=grid_size)
    mask = foreground_mask_from_coverage(coverage, min_coverage=min_coverage)
    return mask, coverage


if __name__ == "__main__":
    import sys

    from preprocess import load_and_preprocess

    if len(sys.argv) != 2:
        print("Usage: python mask.py <path-to-signature-image>")
        raise SystemExit(1)

    _, preprocessed = load_and_preprocess(sys.argv[1])
    fg_mask, coverage = compute_foreground_mask(preprocessed)
    print(f"Grid shape: {fg_mask.shape}")
    print(f"Foreground cells: {fg_mask.sum()} / {fg_mask.size} "
          f"({100 * fg_mask.sum() / fg_mask.size:.1f}%)")
    print(f"Coverage min/mean/max: {coverage.min():.3f}/{coverage.mean():.3f}/{coverage.max():.3f}")


In [ ]:
%%writefile /kaggle/working/densecl_src/augment.py
"""Stroke-safe augmentations for producing DenseCL's two independent views.

Augmentation runs on the RAW grayscale original (dark ink, light/white
background - ink near 0, paper near 255), *before* the tight
crop-to-ink-bbox step in `preprocess_signature`, not after it. Warping an
already tightly-cropped 256x256 image was the original design here, but
it has no room: a tightly-bound signature has strokes sitting right at
the canvas edge, and rotating it pushes those strokes past a fixed-size
output frame with nowhere to go, permanently clipping real content -
padding the *output* canvas doesn't fix this, since cropping back down to
the original footprint afterward throws the displaced content away
regardless of how much blank padding surrounded it during the warp.

Augmenting the raw original first, keeping the canvas generously padded
throughout (never cropping back down inside this module), and only
letting the *existing* `preprocess_signature` crop tightly *after*
augmentation is done fixes this: the tight crop is always computed from
the post-augmentation content, so it can never cut anything off.

Two stages, split at exactly the one point where binarization is safe to
do (see `add_boundary_noise` for why):

  Stage 1 (raw grayscale, ink=low/dark, background=high/light - matches
  a raw scan): `augment_view` chains random occlusion (`random_cutout`),
  geometric warps (`random_affine`, `elastic_warp`), and stroke-width
  jitter (`dilate_strokes`, `erode_strokes`, `random_morphology`). Output
  is unbinarized and usually larger than the input, since the geometric
  warps enlarge the canvas rather than cropping back down - nothing is
  thresholded or cropped tight until `preprocess_signature` runs on it.

  Stage 2 (post `preprocess_signature`, ink=255/background=0 - the
  opposite polarity from stage 1): `add_boundary_noise` jitters stroke
  edges on the final, already tightly-cropped 256x256 binary image.

`generate_view` chains all three (`augment_view` -> `preprocess_signature`
-> `add_boundary_noise`) into the one function that produces a complete
training view from a raw original.

Deliberately excludes anything meant for continuous-tone photos (color
jitter, solarization, blur, aggressive random crop) since those are
either meaningless or destructive on a signature scan.
"""

from __future__ import annotations

import math
from dataclasses import dataclass

import cv2
import numpy as np

from preprocess import otsu_binarize, preprocess_signature

PAPER = 255  # background fill value for a raw grayscale scan (light paper)


def _pad_canvas(image: np.ndarray, pad: int) -> np.ndarray:
    """Add a `pad`-pixel blank (paper-white) border on every side."""
    return cv2.copyMakeBorder(image, pad, pad, pad, pad, borderType=cv2.BORDER_CONSTANT, value=PAPER)


@dataclass(frozen=True)
class AugmentConfig:
    """Tunable parameters for stroke-safe augmentation.

    Defaults are starting points, not final values - meant to be swept
    the same way `w_fg` (Experiment 1) and `min_coverage` were, once a
    downstream validation signal exists to judge them against.

    Strengthened once already (see chat): the first-pass defaults below
    (rotation 8 deg, scale +-8%, shear 6 deg, elastic alpha 8/sigma 6,
    morph_kernel 3, noise_std 25) turned out to be too gentle - visually,
    only dilation produced a clearly "hard" example; rotation/shear/elastic/
    noise were barely visible after the eventual resize-to-256. The goal
    of this augmentation is NOT to mimic how much a real signature
    naturally varies between two genuine samples - it's to force the
    encoder to discover invariant stroke structure by making the pretext
    task genuinely hard, the same logic DenseCL/MoCo/SimCLR use on natural
    images (whose augmentations go well beyond a single photo's natural
    variation). The only real ceiling is "don't distort so far the
    signature becomes a different topology" (loops merging, strokes
    shattering) - see `morph_kernel`'s note.
    """

    rotation_deg: float = 18.0         # max abs rotation, degrees (was 8.0)
    scale_range: tuple[float, float] = (0.85, 1.15)  # min/max scale factor (was 0.92-1.08)
    shear_deg: float = 11.0            # max abs shear angle, degrees (was 6.0)

    elastic_alpha: float = 18.0        # max pixel displacement magnitude (was 8.0)
    elastic_sigma: float = 9.0         # smoothness of the displacement field (was 6.0)
    # alpha and sigma raised TOGETHER, not alpha alone: sigma controls how
    # spatially smooth the displacement field is, and signature strokes are
    # only ~2-4px wide at 256 resolution - a bigger alpha without a
    # correspondingly bigger sigma risks a jagged, high-frequency warp that
    # tears strokes apart rather than flexing them, the same failure mode
    # erosion had (see morph_prob's note below).

    morph_kernel: int = 5              # structuring element side length, odd (was 3)
    # Ceiling to watch for: a kernel this size can fuse separate nearby
    # strokes together (e.g. closing a loop that should stay open) rather
    # than just thickening them - a topology change, not just a thickness
    # change. Re-verified visually (gallery) that this doesn't happen at 5;
    # would need to drop back to 3 if a stronger effect were needed and
    # fusing appeared instead.
    morph_prob: float = 0.35           # probability dilation (thickening) is applied
    # (0.35, not the original 0.7: that was the chance of *either* thicken
    # or thin firing, 50/50 - now that random_morphology only thickens,
    # keeping 0.7 would silently double how often it fires. 0.35 keeps
    # the per-direction rate the same as before erosion was dropped.)

    noise_std: float = 35.0            # gaussian noise std, 0-255 scale (was 25.0)
    # Deliberately still the weakest lever here, by design, not an
    # oversight: this runs last, on the already-clean binary image, and
    # gets partially undone by its own immediate re-threshold - its job is
    # fine boundary jitter (mimicking scan/print roughness), not a primary
    # difficulty driver. Rotation/shear/elastic/morphology do that work.

    cutout_prob: float = 0.3           # probability a cutout pass is applied at all
    cutout_count: int = 1              # number of erased patches per application
    cutout_size_range: tuple[float, float] = (0.05, 0.15)  # patch side, as a fraction of the image's shorter side
    # New augmentation (see chat): small random rectangular patches erased
    # to paper-white, simulating pen skips / faded strokes / overlapping
    # ink - a well-established general technique (often called "Cutout" /
    # "Random Erasing" in the wider computer-vision literature), not
    # previously used in this pipeline. Runs FIRST in `augment_view` (see
    # that function), before any geometric warp, so the erased patch
    # itself gets naturally rotated/warped along with everything else
    # rather than looking like an axis-aligned rectangle stamped on top of
    # an already-warped image. Sized relative to the RAW original's own
    # dimensions (not the later padded/warped canvas), so it reliably lands
    # somewhere near the actual signature content rather than in blank
    # padding by chance. Specifically targets the dense branch: forces
    # patch-level correspondence to hold even when part of a stroke's
    # local neighborhood is missing, a more targeted "hard example" for
    # DenseCL's actual matching mechanism than the whole-image geometric
    # transforms are.


DEFAULT_CONFIG = AugmentConfig()


def _affine_padding(h: int, w: int, config: AugmentConfig) -> int:
    """Border (px) needed so rotation/scale/shear can't push ink off-canvas.

    Sized from the worst-case combined displacement of the farthest
    corner point under the configured rotation/scale/shear ranges, for
    *this specific image's* own dimensions - raw originals vary a lot in
    size and aspect ratio across datasets (BHSig260 scans run ~279x950,
    for instance), so this can't be a fixed constant.
    """
    half_diag = math.sqrt((h / 2.0) ** 2 + (w / 2.0) ** 2)
    rotation_shift = half_diag * math.sin(math.radians(config.rotation_deg))
    scale_shift = half_diag * (max(config.scale_range) - 1.0)
    shear_shift = half_diag * math.tan(math.radians(config.shear_deg))
    return int(math.ceil(rotation_shift + scale_shift + shear_shift)) + 8  # safety buffer


def _elastic_padding(config: AugmentConfig) -> int:
    """Border (px) needed so the elastic displacement field can't push ink off-canvas."""
    return int(math.ceil(config.elastic_alpha)) + 8


def cutout_strokes(image: np.ndarray, rng: np.random.Generator, config: AugmentConfig = DEFAULT_CONFIG) -> np.ndarray:
    """Unconditionally erase `cutout_count` small random rectangular
    patches to paper-white on the RAW original. `rng` is still used - to
    choose WHERE the patch(es) land - but this always applies at least one
    erasure; `random_cutout` (below) is the probability-gated wrapper that
    decides WHETHER to call this at all, same split as `dilate_strokes`
    (unconditional effect) vs. `random_morphology` (its gated wrapper).

    Only ever removes ink, never adds any - unlike the noise-before-crop
    bug this pipeline already fixed once (`add_boundary_noise`'s
    docstring), erasing a small patch can only shrink the eventual ink
    bounding box slightly, never blow it out, so this is safe to run
    pre-crop alongside the other `augment_view` steps.

    Placement is confined to the actual ink bounding box (via
    `otsu_binarize`, same convention as `preprocess.py`'s own bbox step),
    NOT sampled uniformly over the whole raw image - fixed a real bug found
    by regression-testing this exact function: raw scans have substantial
    blank paper margin around the signature, so a patch placed uniformly
    over the full image landed on already-blank background in ~80% of the
    trials where it "fired" (only 3/18 fired attempts actually touched ink).

    Bbox-confinement alone wasn't enough, though (re-tested: only improved
    to 6/18) - a signature is sparse even WITHIN its own tight bounding box
    (loops, gaps between letters/words; measured ~12% ink density there).
    So placement additionally uses rejection sampling: try up to 10 random
    positions within the bbox and keep the first one that actually
    overlaps ink, falling back to the last attempt in the rare case none
    do. This is what makes a "fired" cutout reliably occlude real stroke
    content instead of silently landing on blank space most of the time.
    """
    out = image.copy()
    h, w = out.shape

    ink_mask = otsu_binarize(image) > 0
    ys, xs = np.nonzero(ink_mask)
    if ys.size == 0:
        return out  # blank/corrupt scan - nothing to occlude

    y_min, y_max = int(ys.min()), int(ys.max())
    x_min, x_max = int(xs.min()), int(xs.max())
    # Reference the bbox's LONGER extent, not the shorter one: `_square_pad`
    # pads the shorter side up to match the longer one, then resizes that
    # square down to 256 - so it's the longer extent that sets the eventual
    # downscale factor. Sizing against the shorter extent instead (an
    # earlier version of this function did) made cutout patches shrink to
    # just a handful of pixels after the final resize for elongated bboxes
    # like BHSig260's (short, wide signatures) - confirmed empirically via
    # the gallery: some patches ended up as small as 1-4 changed pixels in
    # the final 256x256 image, nowhere near the intended 5-15%.
    reference_extent = max(max(y_max - y_min, x_max - x_min), 1)

    for _ in range(config.cutout_count):
        size = max(1, int(rng.uniform(*config.cutout_size_range) * reference_extent))
        y_range = max(y_max - y_min - size, 1)
        x_range = max(x_max - x_min - size, 1)

        chosen_y0, chosen_x0 = None, None
        for _attempt in range(10):
            y0 = min(y_min + int(rng.integers(0, y_range)), h - size)
            x0 = min(x_min + int(rng.integers(0, x_range)), w - size)
            if chosen_y0 is None:
                chosen_y0, chosen_x0 = y0, x0  # fallback if no attempt overlaps ink
            if ink_mask[y0:y0 + size, x0:x0 + size].any():
                chosen_y0, chosen_x0 = y0, x0
                break

        out[chosen_y0:chosen_y0 + size, chosen_x0:chosen_x0 + size] = PAPER

    return out


def random_cutout(image: np.ndarray, rng: np.random.Generator, config: AugmentConfig = DEFAULT_CONFIG) -> np.ndarray:
    """With probability `cutout_prob`, call `cutout_strokes`; otherwise
    leave the image unchanged. This is what `augment_view` calls."""
    if rng.random() >= config.cutout_prob:
        return image
    return cutout_strokes(image, rng, config)


def random_affine(image: np.ndarray, rng: np.random.Generator, config: AugmentConfig = DEFAULT_CONFIG) -> np.ndarray:
    """Random rotation + scale + shear about the image center.

    Pads the canvas first (see `_affine_padding`) and does NOT crop back
    down afterward - the output is intentionally larger than the input,
    so displaced strokes have somewhere to land instead of being clipped.
    The caller runs `preprocess_signature` on the final result, which
    crops tightly to wherever the ink actually ended up.
    """
    h, w = image.shape
    pad = _affine_padding(h, w, config)
    padded = _pad_canvas(image, pad)
    ph, pw = padded.shape

    angle = rng.uniform(-config.rotation_deg, config.rotation_deg)
    scale = rng.uniform(config.scale_range[0], config.scale_range[1])
    shear = np.tan(np.deg2rad(rng.uniform(-config.shear_deg, config.shear_deg)))

    def to_3x3(m: np.ndarray) -> np.ndarray:
        return np.vstack([m, [0.0, 0.0, 1.0]]).astype(np.float32)

    rot_scale = to_3x3(cv2.getRotationMatrix2D((pw / 2.0, ph / 2.0), angle, scale))
    shear_mat = to_3x3(np.array([[1.0, shear, 0.0], [0.0, 1.0, 0.0]], dtype=np.float32))
    affine_matrix = (rot_scale @ shear_mat)[:2, :]

    return cv2.warpAffine(
        padded, affine_matrix, (pw, ph),
        flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=PAPER,
    )


def elastic_warp(image: np.ndarray, rng: np.random.Generator, config: AugmentConfig = DEFAULT_CONFIG) -> np.ndarray:
    """Smooth random per-pixel displacement field (local stroke wobble).

    Same padded-canvas, no-crop-back treatment as `random_affine`, for
    the same reason.
    """
    h, w = image.shape
    pad = _elastic_padding(config)
    padded = _pad_canvas(image, pad)
    ph, pw = padded.shape

    dx = rng.uniform(-1.0, 1.0, size=(ph, pw)).astype(np.float32)
    dy = rng.uniform(-1.0, 1.0, size=(ph, pw)).astype(np.float32)
    dx = cv2.GaussianBlur(dx, (0, 0), sigmaX=config.elastic_sigma) * config.elastic_alpha
    dy = cv2.GaussianBlur(dy, (0, 0), sigmaX=config.elastic_sigma) * config.elastic_alpha

    x, y = np.meshgrid(np.arange(pw, dtype=np.float32), np.arange(ph, dtype=np.float32))
    map_x = x + dx
    map_y = y + dy

    return cv2.remap(
        padded, map_x, map_y,
        interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=PAPER,
    )


def _stroke_kernel(config: AugmentConfig) -> np.ndarray:
    # Elliptical (cross-like at small sizes), not a full square: a square
    # kernel requires all 8 neighbors (incl. diagonals) to agree, which
    # shatters thin strokes into disconnected speckle instead of just
    # thinning them. The elliptical kernel is the gentler standard choice
    # for thin curved structures.
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (config.morph_kernel, config.morph_kernel))


def dilate_strokes(image: np.ndarray, config: AugmentConfig = DEFAULT_CONFIG) -> np.ndarray:
    """Thicken (dark) ink strokes by one structuring-element radius.

    Ink is the LOW-intensity value here (raw grayscale: dark ink, light
    paper), so growing the dark region means taking the local MINIMUM,
    which is `cv2.erode` - not `cv2.dilate`. (Named for what it does to
    the ink, not for which OpenCV call implements it.)
    """
    return cv2.erode(image, _stroke_kernel(config), iterations=1)


def erode_strokes(image: np.ndarray, config: AugmentConfig = DEFAULT_CONFIG) -> np.ndarray:
    """Thin (dark) ink strokes by one structuring-element radius.

    Thinning the dark ink means growing the light background - the local
    MAXIMUM, i.e. `cv2.dilate` - for the same polarity reason as
    `dilate_strokes` above.
    """
    return cv2.dilate(image, _stroke_kernel(config), iterations=1)


def random_morphology(image: np.ndarray, rng: np.random.Generator, config: AugmentConfig = DEFAULT_CONFIG) -> np.ndarray:
    """With probability `morph_prob`, thicken strokes; otherwise leave unchanged.

    Erosion (thinning) is deliberately excluded from this default policy.
    On strokes that are already only a few pixels wide post-preprocessing,
    even the gentle elliptical kernel used here disconnects them into
    disjoint fragments rather than just thinning them - confirmed
    visually in orchestration.py's "erode" demo gallery. Unlike rotation,
    shear, elastic wobble, or boundary noise, that fragmentation isn't a
    realistic stand-in for real intra-writer variation: a genuinely
    lighter or faster pen stroke stays thin but continuous, it doesn't
    drop out mid-stroke. Training the pretext to match a shattered stroke
    against its intact counterpart teaches a correspondence that has no
    analog in any real genuine/forgery comparison. `erode_strokes` is
    kept as a standalone function, and in the demo gallery, so this
    failure mode stays visible even though it isn't used here.
    """
    if rng.random() >= config.morph_prob:
        return image
    return dilate_strokes(image, config)


def augment_view(image: np.ndarray, rng: np.random.Generator, config: AugmentConfig = DEFAULT_CONFIG) -> np.ndarray:
    """Cutout + geometric + stroke-width jitter stage: random cutout ->
    affine -> elastic warp -> random morphology. Returns a grayscale
    image, generally larger than the input and NOT yet binarized - the
    caller runs `preprocess_signature` on the result to get the final
    clean 256x256 view (see `generate_view`, which does this for you).

    Cutout runs FIRST, before any geometric warp, so the erased patch
    itself gets naturally rotated/warped along with everything else
    rather than appearing as an axis-aligned rectangle stamped onto an
    already-warped image.

    Deliberately does NOT include noise (see `add_boundary_noise` for
    why) or re-threshold internally (see `add_boundary_noise`'s docstring
    for how repeated re-thresholding erased small features like a
    diacritic dot in an earlier version of this pipeline).
    """
    out = random_cutout(image, rng, config)
    out = random_affine(out, rng, config)
    out = elastic_warp(out, rng, config)
    out = random_morphology(out, rng, config)
    return out


def add_boundary_noise(image: np.ndarray, rng: np.random.Generator, config: AugmentConfig = DEFAULT_CONFIG) -> np.ndarray:
    """Jitter stroke boundaries on an already-preprocessed binary image
    (ink=255, background=0) - run this AFTER `preprocess_signature`, not
    as part of the pre-crop `augment_view` chain.

    Noise used to run pre-crop, across the large padded working canvas
    `augment_view` produces. That was actively dangerous: the crop-to-
    bbox step in `preprocess_signature` computes the ink bounding box as
    a plain min/max over thresholded pixel coordinates, which has zero
    outlier robustness. Adding noise across a canvas that's mostly blank
    padding (often several times larger than the final 256x256, since
    e.g. a BHSig260 scan pads out to ~650x1300) creates enough scattered
    false-positive "ink" pixels that the computed bounding box can blow
    out to cover almost the entire padded canvas - confirmed empirically:
    21.5% of a 649x1319 canvas got classified as ink, with a bounding box
    spanning the full frame. Doing noise here instead, on the final,
    already tightly-cropped and fixed-size 256x256 canvas, is safe: there
    is no further cropping downstream for stray pixels to corrupt, and
    the dense, mostly-signature content means Otsu's threshold stays well
    calibrated.
    """
    noise = rng.normal(0.0, config.noise_std, size=image.shape).astype(np.float32)
    noisy = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    # Plain THRESH_BINARY, not INV: `image` is already in the ink=255/
    # background=0 convention (preprocess_signature's output), the
    # opposite polarity from the raw-scan convention the rest of this
    # module works in.
    _, binary = cv2.threshold(noisy, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary


def generate_view(image: np.ndarray, rng: np.random.Generator, config: AugmentConfig = DEFAULT_CONFIG) -> np.ndarray:
    """End-to-end: raw grayscale original -> one independently-augmented,
    clean 256x256 binary view (ink=255, background=0).

    This is the function View A / View B generation should call: it
    chains `augment_view` (geometric warps + stroke-width jitter on the
    raw original), `preprocess_signature` (the one tight crop and the
    one primary binarization), and `add_boundary_noise` (a second,
    deliberately-last, safe-to-repeat threshold pass for boundary
    jitter). Call this twice with the same `rng` (which advances between
    calls) to get View A and View B for the same source signature.
    """
    raw = augment_view(image, rng, config)
    clean = preprocess_signature(raw)
    return add_boundary_noise(clean, rng, config)


if __name__ == "__main__":
    import sys

    if len(sys.argv) != 2:
        print("Usage: python augment.py <path-to-signature-image>")
        raise SystemExit(1)

    original = cv2.imread(sys.argv[1], cv2.IMREAD_GRAYSCALE)
    if original is None:
        raise FileNotFoundError(sys.argv[1])

    rng = np.random.default_rng(0)
    view_a = generate_view(original, rng, DEFAULT_CONFIG)
    view_b = generate_view(original, rng, DEFAULT_CONFIG)

    print(f"Original shape: {original.shape}")
    print(f"View A: {view_a.shape}, ink px={int((view_a > 0).sum())}")
    print(f"View B: {view_b.shape}, ink px={int((view_b > 0).sum())}")


In [ ]:
%%writefile /kaggle/working/densecl_src/test_set_creation.py
"""Writer-level test-set split - Kaggle version.

Identical logic to the local project's `test_set_creation.py`. The only
change: `DATA_ROOT` comes from `kaggle_config.py` (the notebook's single
editable config cell) instead of being derived from this file's own location
on disk, and `OUTPUT_DIR` points at a fixed path under the flat Kaggle
source directory. The actual held-out writer split is NOT regenerated here -
the notebook writes the already-frozen split JSONs (identical to the ones
already used for the local training run) directly to `OUTPUT_DIR` in an
earlier cell, so `main()`/`build_test_split()`/`save_test_split()` below are
kept only for completeness (e.g. if you ever need to rebuild a split from
scratch on new data) and are not invoked by the training pipeline.
"""

from __future__ import annotations

import argparse
import json
import random
from pathlib import Path

from kaggle_config import DATA_ROOT, SEED

OUTPUT_DIR = Path("/kaggle/working/densecl_src/data/test_set_writer_split")

# dataset folder name (under DATA_ROOT/) -> number of writers to hold out for testing.
# Mendeley is intentionally absent - it has no genuine/forged labels, see
# validation_set_creation.py's docstring for the full reasoning.
TEST_WRITER_COUNTS: dict[str, int] = {
    "CEDAR": 15,
    "BHSig260_Bengali": 30,
    "BHSig260_Hindi": 30,
}


def list_writer_ids(dataset_dir: Path) -> list[str]:
    """Every writer directory name under a dataset folder, as strings (writer
    IDs are numeric-looking but not treated as ints - Mendeley's writer
    folders, e.g. `a_(1)`, are not numeric at all)."""
    return sorted(p.name for p in dataset_dir.iterdir() if p.is_dir())


def select_test_writers(writer_ids: list[str], num_test: int, seed: int) -> list[str]:
    """Randomly choose `num_test` writer IDs to hold out, using a fixed seed
    so the choice is reproducible if the split is intentionally rebuilt from
    scratch. Returned sorted purely for a readable, diff-friendly JSON file -
    sorting the OUTPUT does not affect the randomness of the selection."""
    if num_test > len(writer_ids):
        raise ValueError(
            f"Requested {num_test} test writers but only {len(writer_ids)} writers exist"
        )
    rng = random.Random(seed)
    return sorted(rng.sample(writer_ids, num_test))


def build_test_split(dataset_name: str, num_test: int, seed: int = SEED) -> dict:
    dataset_dir = DATA_ROOT / dataset_name
    if not dataset_dir.is_dir():
        raise FileNotFoundError(f"Dataset folder not found: {dataset_dir}")

    writer_ids = list_writer_ids(dataset_dir)
    test_writer_ids = select_test_writers(writer_ids, num_test, seed)

    return {
        "dataset": dataset_name,
        "total_writers": len(writer_ids),
        "num_test_writers": len(test_writer_ids),
        "seed": seed,
        "test_writer_ids": test_writer_ids,
    }


def save_test_split(split: dict, output_dir: Path = OUTPUT_DIR, overwrite: bool = False) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / f"{split['dataset']}_test_writers.json"

    if out_path.exists() and not overwrite:
        raise FileExistsError(
            f"{out_path} already exists. Refusing to overwrite by default - a test split, "
            f"once used for any training run, must never silently change. Pass "
            f"overwrite=True / --overwrite only if you are certain no training has happened "
            f"against the existing split yet."
        )

    with open(out_path, "w") as f:
        json.dump(split, f, indent=2)
    return out_path


def load_test_writer_ids(dataset_name: str, split_dir: Path = OUTPUT_DIR) -> set[str]:
    """Read back a previously-created split - the function any SSL/downstream
    dataset-building code should call to exclude held-out test writers.
    Returns an empty set for a dataset with no split file (e.g. Mendeley),
    meaning "hold out nothing," which is the correct behavior for it."""
    split_path = split_dir / f"{dataset_name}_test_writers.json"
    if not split_path.exists():
        return set()
    with open(split_path) as f:
        return set(json.load(f)["test_writer_ids"])


def main(overwrite: bool = False) -> None:
    for dataset_name, num_test in TEST_WRITER_COUNTS.items():
        split = build_test_split(dataset_name, num_test)
        out_path = save_test_split(split, overwrite=overwrite)
        print(f"[{dataset_name}] {split['num_test_writers']}/{split['total_writers']} writers "
              f"held out for testing -> {out_path}")

    print("[Mendeley] no usable genuine/forged labels - 0 writers held out, "
          "entire dataset used for SSL pretraining only (no split file created).")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Create writer-level test splits per dataset.")
    parser.add_argument("--overwrite", action="store_true",
                         help="Allow overwriting an existing split file. Only use this if no "
                              "training has happened against the existing split yet.")
    args = parser.parse_args()
    main(overwrite=args.overwrite)


In [ ]:
%%writefile /kaggle/working/densecl_src/validation_set_creation.py
"""Writer-level SSL validation split - Kaggle version.

Identical logic to the local project's `validation_set_creation.py`. The
only change: `OUTPUT_DIR` points at a fixed path under the flat Kaggle
source directory instead of a `Path(__file__)`-derived one; `DATA_ROOT`/
`SEED` are inherited transitively from `test_set_creation_kaggle.py`
(itself sourced from `kaggle_config.py`). As with the test split, the
notebook writes the already-frozen validation split JSONs directly to
`OUTPUT_DIR` in an earlier cell - `main()` below is kept only for
completeness and is not invoked by the training pipeline.

Companion to `test_set_creation.py`, but for a DIFFERENT purpose and with a
DIFFERENT inclusion rule: this split tracks the unlabeled SSL contrastive
loss (global + dense) on writers the pretraining loop never trains on, to
catch the SSL-specific overfitting failure mode (the encoder exploiting
training-image-specific quirks rather than learning generalizable stroke
structure) - since it never touches labels, Mendeley IS included here
(unlike the test split, which excludes it for having no usable labels).
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

from test_set_creation import (
    DATA_ROOT,
    SEED,
    list_writer_ids,
    load_test_writer_ids,
)
from test_set_creation import select_test_writers as _select_random_writers

OUTPUT_DIR = Path("/kaggle/working/densecl_src/data/validation_set_writer_split")

VALIDATION_WRITER_COUNTS: dict[str, int] = {
    "CEDAR": 5,
    "BHSig260_Bengali": 10,
    "BHSig260_Hindi": 15,
    "Mendeley": 20,
}


def build_validation_split(dataset_name: str, num_validation: int, seed: int = SEED) -> dict:
    dataset_dir = DATA_ROOT / dataset_name
    if not dataset_dir.is_dir():
        raise FileNotFoundError(f"Dataset folder not found: {dataset_dir}")

    all_writer_ids = list_writer_ids(dataset_dir)
    test_writer_ids = load_test_writer_ids(dataset_name)  # empty set for Mendeley
    eligible_writer_ids = [w for w in all_writer_ids if w not in test_writer_ids]

    validation_writer_ids = _select_random_writers(eligible_writer_ids, num_validation, seed)

    return {
        "dataset": dataset_name,
        "total_writers": len(all_writer_ids),
        "num_test_writers_excluded": len(test_writer_ids),
        "num_eligible_writers": len(eligible_writer_ids),
        "num_validation_writers": len(validation_writer_ids),
        "seed": seed,
        "validation_writer_ids": validation_writer_ids,
    }


def save_validation_split(split: dict, output_dir: Path = OUTPUT_DIR, overwrite: bool = False) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / f"{split['dataset']}_validation_writers.json"

    if out_path.exists() and not overwrite:
        raise FileExistsError(
            f"{out_path} already exists. Refusing to overwrite by default - a validation "
            f"split, once used for any training run, must never silently change. Pass "
            f"overwrite=True / --overwrite only if you are certain no training has happened "
            f"against the existing split yet."
        )

    with open(out_path, "w") as f:
        json.dump(split, f, indent=2)
    return out_path


def load_validation_writer_ids(dataset_name: str, split_dir: Path = OUTPUT_DIR) -> set[str]:
    """Read back a previously-created split. Returns an empty set if no
    split file exists for this dataset yet."""
    split_path = split_dir / f"{dataset_name}_validation_writers.json"
    if not split_path.exists():
        return set()
    with open(split_path) as f:
        return set(json.load(f)["validation_writer_ids"])


def main(overwrite: bool = False) -> None:
    for dataset_name, num_validation in VALIDATION_WRITER_COUNTS.items():
        split = build_validation_split(dataset_name, num_validation)
        out_path = save_validation_split(split, overwrite=overwrite)
        print(f"[{dataset_name}] {split['num_validation_writers']}/{split['num_eligible_writers']} eligible "
              f"writers held out for validation (of {split['total_writers']} total, "
              f"{split['num_test_writers_excluded']} already test-excluded) -> {out_path}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Create writer-level SSL validation splits per dataset.")
    parser.add_argument("--overwrite", action="store_true",
                         help="Allow overwriting an existing split file. Only use this if no "
                              "training has happened against the existing split yet.")
    args = parser.parse_args()
    main(overwrite=args.overwrite)


In [ ]:
%%writefile /kaggle/working/densecl_src/dataset.py
"""PyTorch Dataset producing DenseCL's two-view training tuples.

Given a signature image path, `SignatureSSLDataset.__getitem__` returns two
independently, stochastically augmented views of the same signature (View A,
View B) plus each view's foreground grid mask - exactly the four tensors the
DenseCL pretraining loop needs (global + dense losses, foreground-masked).

No labels are used anywhere here: writer identity is not read for training
purposes, only checked against the held-out test split (see below) so
those writers can be excluded. This is a pretraining-stage dataset, not the
downstream verification dataset.

Writer identity IS used for one thing: excluding the held-out test writers
created by `test_set_creation.py`. A writer-independent verifier's test set
must never be seen by EITHER pipeline stage - not just the downstream
supervised stage, the SSL pretraining stage too - or "held out" stops
meaning anything. See `list_all_signature_paths`'s `exclude_test_writers`
parameter (on by default).
"""

from __future__ import annotations

from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

from augment import DEFAULT_CONFIG, AugmentConfig, generate_view
from mask import compute_foreground_mask
from test_set_creation import load_test_writer_ids

# Silences OpenCV's own internal WARNING-level logging (e.g. grfmt_tiff.cpp's
# "TIFFFetchNormalTag: ... Software tag contains null byte" spam - harmless
# metadata truncation on every BHSig260 .tif read, not a real problem) while
# still surfacing actual errors. Set once at import time since this is a
# process-wide OpenCV setting, not per-call.
cv2.utils.logging.setLogLevel(cv2.utils.logging.LOG_LEVEL_ERROR)

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}


def list_all_signature_paths(
    data_root: Path,
    exclude_test_writers: bool = True,
    extra_exclude_writer_ids: dict[str, set[str]] | None = None,
) -> list[Path]:
    """Every signature image path under `data_root/<dataset>/<writer>/<file>`.

    `exclude_test_writers=True` (the default, and the only correct setting
    for an actual training run) skips any writer listed in that dataset's
    held-out test split (`test_set_creation.py` /
    `data/test_set_writer_split/<dataset>_test_writers.json`). Datasets with
    no split file (Mendeley - see `test_set_creation.py`'s docstring for why)
    have nothing excluded, via `load_test_writer_ids`'s empty-set fallback.
    Only pass `False` for debugging/inspection - never for training.

    `extra_exclude_writer_ids`, if given, additionally skips specific writer
    IDs per dataset (e.g. `{"CEDAR": {"3", "7"}}`) - used to also strip the
    SSL validation writers (`validation_set_creation.py`) out of the
    training pool, on top of the test writers.
    """
    data_root = Path(data_root)
    paths = []
    excluded_writer_counts: dict[str, int] = {}

    for dataset_dir in sorted(p for p in data_root.iterdir() if p.is_dir()):
        test_writer_ids = load_test_writer_ids(dataset_dir.name) if exclude_test_writers else set()
        extra_ids = (extra_exclude_writer_ids or {}).get(dataset_dir.name, set())
        skip_ids = test_writer_ids | extra_ids
        excluded_writer_counts[dataset_dir.name] = 0

        for writer_dir in sorted(p for p in dataset_dir.iterdir() if p.is_dir()):
            if writer_dir.name in skip_ids:
                excluded_writer_counts[dataset_dir.name] += 1
                continue
            for image_path in sorted(writer_dir.iterdir()):
                if image_path.suffix.lower() in IMAGE_EXTENSIONS:
                    paths.append(image_path)

    for dataset_name, count in excluded_writer_counts.items():
        if count > 0:
            print(f"[SignatureSSLDataset] {dataset_name}: excluded {count} writer(s) (test and/or validation)")

    return paths


def list_specific_writer_signature_paths(data_root: Path, writer_ids: dict[str, set[str]]) -> list[Path]:
    """Every signature image path, but restricted to ONLY the given writer
    IDs per dataset (e.g. `{"CEDAR": {"3", "7"}}`) - the inverse query shape
    from `list_all_signature_paths` ("only these" instead of "everything
    except these"). Used to build the small, fixed SSL validation pool from
    `validation_set_creation.py`'s output."""
    data_root = Path(data_root)
    paths = []
    for dataset_dir in sorted(p for p in data_root.iterdir() if p.is_dir()):
        wanted = writer_ids.get(dataset_dir.name, set())
        if not wanted:
            continue
        for writer_dir in sorted(p for p in dataset_dir.iterdir() if p.is_dir()):
            if writer_dir.name not in wanted:
                continue
            for image_path in sorted(writer_dir.iterdir()):
                if image_path.suffix.lower() in IMAGE_EXTENSIONS:
                    paths.append(image_path)
    return paths


class SignatureSSLDataset(Dataset):
    """Yields (view_a, view_b, mask_a, mask_b) for DenseCL pretraining.

    `view_a`/`view_b`: float32 tensors, shape (1, 256, 256), values in
    [0, 1] (ink=1.0, background=0.0) - the channel dim matches the
    existing thesis encoder's 1-channel input convention.
    `mask_a`/`mask_b`: bool tensors, shape (32, 32), True where the grid
    cell counts as foreground (see `mask.py`'s `min_coverage` default).

    Each `__getitem__` call draws fresh randomness (a `np.random.default_rng`
    seeded from the item index and the epoch-varying `torch` seed state), so
    the same index yields a different augmented pair on every access - the
    normal SSL behavior of re-augmenting every epoch rather than caching one
    fixed pair per image.
    """

    def __init__(
        self,
        data_root: str | Path,
        config: AugmentConfig = DEFAULT_CONFIG,
        grid_size: int = 32,
        min_coverage: float = 0.0,
        exclude_test_writers: bool = True,
        extra_exclude_writer_ids: dict[str, set[str]] | None = None,
        image_paths_override: list[Path] | None = None,
    ) -> None:
        """`image_paths_override`, if given, bypasses the normal directory
        walk entirely and uses exactly this list of paths - the mechanism
        `driver/train.py` uses to build the validation dataset from
        `list_specific_writer_signature_paths`'s output (an "only these
        writers" pool, not an "everything except" one)."""
        if image_paths_override is not None:
            self.image_paths = image_paths_override
        else:
            self.image_paths = list_all_signature_paths(
                data_root,
                exclude_test_writers=exclude_test_writers,
                extra_exclude_writer_ids=extra_exclude_writer_ids,
            )
        if not self.image_paths:
            raise FileNotFoundError(f"No signature images found under {data_root}")
        self.config = config
        self.grid_size = grid_size
        self.min_coverage = min_coverage

    def __len__(self) -> int:
        return len(self.image_paths)

    def _to_tensor(self, binary_view: np.ndarray) -> torch.Tensor:
        normalized = (binary_view > 0).astype(np.float32)
        return torch.from_numpy(normalized).unsqueeze(0)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        path = self.image_paths[index]
        original = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if original is None:
            raise FileNotFoundError(f"Could not read image at {path}")

        # A fresh, unseeded generator per call: two calls to generate_view
        # below must draw different randomness (else View A == View B), and
        # every subsequent epoch's pass over the same index must also differ.
        rng = np.random.default_rng()

        view_a = generate_view(original, rng, self.config)
        view_b = generate_view(original, rng, self.config)

        mask_a, _ = compute_foreground_mask(view_a, self.grid_size, self.min_coverage)
        mask_b, _ = compute_foreground_mask(view_b, self.grid_size, self.min_coverage)

        return {
            "view_a": self._to_tensor(view_a),
            "view_b": self._to_tensor(view_b),
            "mask_a": torch.from_numpy(mask_a),
            "mask_b": torch.from_numpy(mask_b),
        }


if __name__ == "__main__":
    from pathlib import Path as _Path

    SCRIPT_DIR = _Path(__file__).resolve().parent
    DATA_ROOT = SCRIPT_DIR.parent.parent / "data" / "all"

    dataset = SignatureSSLDataset(DATA_ROOT)
    print(f"Dataset size: {len(dataset)} signature images")

    sample = dataset[0]
    for key, value in sample.items():
        print(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}")

    fg_frac_a = sample["mask_a"].float().mean().item()
    fg_frac_b = sample["mask_b"].float().mean().item()
    print(f"View A foreground fraction: {fg_frac_a:.3f}")
    print(f"View B foreground fraction: {fg_frac_b:.3f}")


In [ ]:
%%writefile /kaggle/working/densecl_src/encoder.py
"""ResNet-style backbone - faithful port of the original thesis encoder.

Ported from `Thesis_Final/ssl_pretraining/models/Encoder.py` and
`Thesis_Final/ssl_pretraining/models/encoder/*.py` (the code that actually
produced the report's numbers - `SelfSupervisedNetwork.__init__` there
instantiates `Encoder(norm_type=norm_type)` with every other argument left
at its default, so the defaults below are not a guess, they're what was
used). This replaces an earlier version of this file that reconstructed the
architecture from the report's prose description alone (stage widths and
downsample placement only - it did not specify a stem or block layout), an
assumption flagged at the time and now resolved by reading the real source.

Two differences from that earlier guess, now corrected:
  - There is a dedicated 2-layer stem (plain `ConvNormAct`, no residual
    connection) that expands 1 -> 32 -> 32 channels before stage 1, not
    folded into stage 1 itself.
  - Stages 2-4 are each exactly one `ProjectionResidualBlock` (the
    stride-2 downsampling block) followed by `num_identity_blocks`
    `IdentityResidualBlock`s (1 by default) - not a generic "first block
    strided, rest identical" pattern with an arbitrary block count.

Architecture, in order: stem (1->32->32, no downsample) -> stage1 (2
identity blocks @ 32, no downsample) -> stage2 (32->64, downsample) ->
stage3 (64->128, downsample) -> stage4 (128->256, downsample). Three
downsamples total, matching report Fig 3.3 (256x256 input -> 32x32x256
dense feature grid), and the same 32x32 resolution `mask.py`'s foreground
grid is computed at.
"""

from __future__ import annotations

import torch
import torch.nn as nn

TARGET_SIZE = 256  # must match preprocess.TARGET_SIZE for the 32x32 grid to hold


def build_norm_layer(num_features: int, norm_type: str = "batch") -> nn.Module:
    if norm_type == "batch":
        return nn.BatchNorm2d(num_features)
    if norm_type == "instance":
        return nn.InstanceNorm2d(num_features, affine=True)
    raise ValueError(f"Unsupported norm_type: {norm_type}")


class ConvNormAct(nn.Module):
    """Conv -> norm -> activation. Used only by the stem (no residual connection)."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int | None = None,
        norm_type: str = "batch",
        activation_layer: type[nn.Module] = nn.ReLU,
    ) -> None:
        super().__init__()
        if padding is None:
            padding = kernel_size // 2

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=False),
            build_norm_layer(out_channels, norm_type=norm_type),
            activation_layer(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class SignatureStem(nn.Module):
    """Two plain ConvNormAct layers, no downsample: 1 -> 32 -> 32 channels by default."""

    def __init__(
        self,
        in_channels: int = 1,
        stem_channels: tuple[int, ...] = (32, 32),
        norm_type: str = "batch",
        activation_layer: type[nn.Module] = nn.ReLU,
    ) -> None:
        super().__init__()
        blocks = []
        current_in = in_channels
        for current_out in stem_channels:
            blocks.append(
                ConvNormAct(current_in, current_out, kernel_size=3, stride=1,
                             norm_type=norm_type, activation_layer=activation_layer)
            )
            current_in = current_out

        self.layers = nn.Sequential(*blocks)
        self.out_channels = stem_channels[-1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)


class IdentityResidualBlock(nn.Module):
    """Two 3x3 convs at a fixed channel count, plain identity skip (no projection)."""

    def __init__(self, in_channels: int, norm_type: str = "batch", activation_layer: type[nn.Module] = nn.ReLU) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.norm1 = build_norm_layer(in_channels, norm_type=norm_type)
        self.act1 = activation_layer(inplace=True)

        self.conv2 = nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.norm2 = build_norm_layer(in_channels, norm_type=norm_type)

        self.out_act = activation_layer(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        out = self.act1(self.norm1(self.conv1(x)))
        out = self.norm2(self.conv2(out))
        return self.out_act(out + residual)


class ProjectionResidualBlock(nn.Module):
    """Stride-2, channel-changing residual block: 1x1-conv shortcut, two 3x3 convs on the main path."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 2,
        norm_type: str = "batch",
        activation_layer: type[nn.Module] = nn.ReLU,
    ) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.norm1 = build_norm_layer(out_channels, norm_type=norm_type)
        self.act1 = activation_layer(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.norm2 = build_norm_layer(out_channels, norm_type=norm_type)

        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
            build_norm_layer(out_channels, norm_type=norm_type),
        )
        self.out_act = activation_layer(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.shortcut(x)
        out = self.act1(self.norm1(self.conv1(x)))
        out = self.norm2(self.conv2(out))
        return self.out_act(out + residual)


class ResidualStage(nn.Module):
    """Stage 1: `num_blocks` IdentityResidualBlocks at a fixed channel count, no downsample."""

    def __init__(self, channels: int, num_blocks: int = 2, norm_type: str = "batch", activation_layer: type[nn.Module] = nn.ReLU) -> None:
        super().__init__()
        self.blocks = nn.Sequential(*[
            IdentityResidualBlock(channels, norm_type=norm_type, activation_layer=activation_layer)
            for _ in range(num_blocks)
        ])
        self.out_channels = channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.blocks(x)


class TransitionResidualStage(nn.Module):
    """Stages 2-4: one ProjectionResidualBlock (downsample + channel change) then `num_identity_blocks` IdentityResidualBlocks."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 2,
        num_identity_blocks: int = 1,
        norm_type: str = "batch",
        activation_layer: type[nn.Module] = nn.ReLU,
    ) -> None:
        super().__init__()
        blocks = [ProjectionResidualBlock(in_channels, out_channels, stride=stride,
                                           norm_type=norm_type, activation_layer=activation_layer)]
        for _ in range(num_identity_blocks):
            blocks.append(IdentityResidualBlock(out_channels, norm_type=norm_type, activation_layer=activation_layer))

        self.blocks = nn.Sequential(*blocks)
        self.out_channels = out_channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.blocks(x)


class Encoder(nn.Module):
    """The full backbone: stem -> stage1 -> stage2 -> stage3 -> stage4.

    `forward(x, pool)`: `pool=False` returns the dense `(batch, 256, 32, 32)`
    feature grid (what the DenseCL dense head and the foreground masks need);
    `pool=True` returns a globally-averaged `(batch, 256)` vector (what a
    global head - or the existing thesis's frozen-encoder downstream stage -
    needs). Matches the original `Encoder.forward` interface exactly, so
    this module is a drop-in match for how the thesis's own code calls it.
    """

    def __init__(
        self,
        in_channels: int = 1,
        stem_channels: tuple[int, ...] = (32, 32),
        stage1_blocks: int = 2,
        stage2_out_channels: int = 64,
        stage2_identity_blocks: int = 1,
        stage3_out_channels: int = 128,
        stage3_identity_blocks: int = 1,
        stage4_out_channels: int = 256,
        stage4_identity_blocks: int = 1,
        norm_type: str = "batch",
    ) -> None:
        super().__init__()

        self.stem = SignatureStem(in_channels=in_channels, stem_channels=stem_channels, norm_type=norm_type)

        self.stage1 = ResidualStage(channels=self.stem.out_channels, num_blocks=stage1_blocks, norm_type=norm_type)

        self.stage2 = TransitionResidualStage(
            in_channels=self.stage1.out_channels, out_channels=stage2_out_channels,
            stride=2, num_identity_blocks=stage2_identity_blocks, norm_type=norm_type,
        )
        self.stage3 = TransitionResidualStage(
            in_channels=self.stage2.out_channels, out_channels=stage3_out_channels,
            stride=2, num_identity_blocks=stage3_identity_blocks, norm_type=norm_type,
        )
        self.stage4 = TransitionResidualStage(
            in_channels=self.stage3.out_channels, out_channels=stage4_out_channels,
            stride=2, num_identity_blocks=stage4_identity_blocks, norm_type=norm_type,
        )

        self.out_channels = self.stage4.out_channels
        self.feature_dim = self.out_channels  # alias used elsewhere in this project (dataset.py, momentum.py)

    def forward(self, x: torch.Tensor, pool: bool = False) -> torch.Tensor:
        if x.ndim != 4:
            raise ValueError(f"Expected input shape (batch, channels, H, W), got {tuple(x.shape)}")
        out = self.stem(x)
        out = self.stage1(out)
        out = self.stage2(out)
        out = self.stage3(out)
        out = self.stage4(out)
        if pool:
            out = torch.mean(out, dim=(2, 3))
        return out


if __name__ == "__main__":
    encoder = Encoder()
    dummy = torch.zeros(2, 1, TARGET_SIZE, TARGET_SIZE)

    dense = encoder(dummy, pool=False)
    pooled = encoder(dummy, pool=True)

    num_params = sum(p.numel() for p in encoder.parameters())
    expected_grid = TARGET_SIZE // 8  # three stride-2 downsamples

    print(f"Input shape:  {tuple(dummy.shape)}")
    print(f"Dense output shape:  {tuple(dense.shape)} (expected grid size {expected_grid}x{expected_grid})")
    print(f"Pooled output shape: {tuple(pooled.shape)}")
    print(f"Parameter count: {num_params:,}")


In [ ]:
%%writefile /kaggle/working/densecl_src/momentum.py
"""Momentum (EMA) twin of `Encoder`, MoCo-style.

The online encoder is trained by backprop as usual. The momentum encoder is
a separate copy that never receives gradients - after each training step,
its weights are nudged slightly toward the online encoder's current weights
via an exponential moving average (EMA):

    momentum_weight <- momentum * momentum_weight + (1 - momentum) * online_weight

With `momentum` close to 1 (default 0.999), the momentum encoder changes
very slowly step to step, which is what makes it useful as a stable target
for View B's features and for the vectors pushed into the memory queue -
see `progress_so_far.md` Section 2 / the architecture discussion for why a
single, fast-changing encoder for both views would give the contrastive
loss a moving target.
"""

from __future__ import annotations

import copy

import torch
import torch.nn as nn

from encoder import Encoder

DEFAULT_MOMENTUM = 0.999


class MomentumEncoder(nn.Module):
    """Wraps a frozen deep copy of an `Encoder`, updated only via EMA."""

    def __init__(self, online_encoder: Encoder) -> None:
        super().__init__()
        self.encoder = copy.deepcopy(online_encoder)
        for param in self.encoder.parameters():
            param.requires_grad_(False)
        self.feature_dim = self.encoder.feature_dim

    @torch.no_grad()
    def forward(self, x: torch.Tensor, pool: bool = False) -> torch.Tensor:
        return self.encoder(x, pool=pool)

    @torch.no_grad()
    def update(self, online_encoder: Encoder, momentum: float = DEFAULT_MOMENTUM) -> None:
        """EMA-update this encoder's weights toward `online_encoder`'s current weights.

        Call this once per training step, after the optimizer step on
        `online_encoder`. Trainable parameters are EMA-updated; BatchNorm
        running statistics (buffers) are copied directly rather than
        EMA-blended - they already track a running average internally, so
        blending them again would just double-smooth the same statistic.
        """
        for param_m, param_o in zip(self.encoder.parameters(), online_encoder.parameters()):
            param_m.data.mul_(momentum).add_(param_o.data, alpha=1.0 - momentum)
        for buffer_m, buffer_o in zip(self.encoder.buffers(), online_encoder.buffers()):
            buffer_m.data.copy_(buffer_o.data)


class EMAModule(nn.Module):
    """Generic momentum (EMA) twin of any `nn.Module` - same mechanism as
    `MomentumEncoder` above, generalized so `heads.py`'s `GlobalHead` and
    `DenseHead` (which take no `pool` argument, unlike `Encoder`) can reuse
    it instead of duplicating the EMA update math a second and third time.
    `MomentumEncoder` is kept as its own class rather than refactored onto
    this, since it's already verified working and its `pool`-forwarding
    `forward` signature is encoder-specific.
    """

    def __init__(self, online_module: nn.Module) -> None:
        super().__init__()
        self.module = copy.deepcopy(online_module)
        for param in self.module.parameters():
            param.requires_grad_(False)

    @torch.no_grad()
    def forward(self, *args, **kwargs):
        return self.module(*args, **kwargs)

    @torch.no_grad()
    def update(self, online_module: nn.Module, momentum: float = DEFAULT_MOMENTUM) -> None:
        for param_m, param_o in zip(self.module.parameters(), online_module.parameters()):
            param_m.data.mul_(momentum).add_(param_o.data, alpha=1.0 - momentum)
        for buffer_m, buffer_o in zip(self.module.buffers(), online_module.buffers()):
            buffer_m.data.copy_(buffer_o.data)


if __name__ == "__main__":
    online = Encoder()
    momentum_encoder = MomentumEncoder(online)

    trainable = sum(p.requires_grad for p in momentum_encoder.parameters())
    print(f"Momentum encoder trainable parameters: {trainable} (expected 0)")

    # Snapshot one weight, perturb the online encoder (simulating an
    # optimizer step), then confirm the EMA update moves the momentum
    # encoder's weight partway toward it - and not all the way, since
    # momentum < 1.
    before = momentum_encoder.encoder.stage1.blocks[0].conv1.weight.clone()
    with torch.no_grad():
        online.stage1.blocks[0].conv1.weight.add_(1.0)  # large perturbation for a visible effect
    momentum_encoder.update(online, momentum=0.9)
    after = momentum_encoder.encoder.stage1.blocks[0].conv1.weight

    max_shift = (after - before).abs().max().item()
    print(f"Max weight shift after one EMA update (momentum=0.9): {max_shift:.4f} (expected ~0.1, i.e. (1-momentum)*1.0)")

    dummy = torch.zeros(2, 1, 256, 256)
    dense_out = momentum_encoder(dummy, pool=False)
    pooled_out = momentum_encoder(dummy, pool=True)
    print(f"Momentum encoder dense output shape:  {tuple(dense_out.shape)}")
    print(f"Momentum encoder pooled output shape: {tuple(pooled_out.shape)}")
    print(f"Output requires_grad: {dense_out.requires_grad} (expected False)")


In [ ]:
%%writefile /kaggle/working/densecl_src/heads.py
"""Projection heads: the small networks the contrastive losses are computed
on, kept separate from the encoder so the encoder's own (pooled/dense)
output - what gets frozen and reused downstream - isn't the thing directly
optimized against the pretext task's augmentation-invariance pressure.

Design (see the architecture discussion in `progress_so_far.md` Section 2):
a 2-layer nonlinear MLP, matching standard MoCo v2 / SimCLR practice, with
the output L2-normalized so cosine similarity (what the contrastive losses
compare with) is well-defined. `GlobalHead` runs on the encoder's pooled
vector; `DenseHead` runs the same MLP independently at every grid cell of
the encoder's dense feature map, via a 1x1 convolution.
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

IN_DIM = 256       # must match Encoder.out_channels / feature_dim
HIDDEN_DIM = 256
PROJECTION_DIM = 128  # MoCo/SimCLR convention - tunable, not load-bearing


class GlobalHead(nn.Module):
    """Pooled (batch, IN_DIM) encoder vector -> L2-normalized (batch, PROJECTION_DIM)."""

    def __init__(self, in_dim: int = IN_DIM, hidden_dim: int = HIDDEN_DIM, out_dim: int = PROJECTION_DIM) -> None:
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, pooled_features: torch.Tensor) -> torch.Tensor:
        if pooled_features.ndim != 2:
            raise ValueError(f"Expected shape (batch, {IN_DIM}), got {tuple(pooled_features.shape)}")
        projected = self.mlp(pooled_features)
        return F.normalize(projected, dim=1)


class DenseHead(nn.Module):
    """Dense (batch, IN_DIM, H, W) encoder grid -> L2-normalized (batch, PROJECTION_DIM, H, W).

    Implemented as two 1x1 convolutions (equivalent to applying the same
    small MLP independently to every grid cell's channel vector - a 1x1
    conv's weights are shared across spatial positions, so this is not a
    different network per cell, just the same one applied everywhere).
    """

    def __init__(self, in_channels: int = IN_DIM, hidden_channels: int = HIDDEN_DIM, out_channels: int = PROJECTION_DIM) -> None:
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, out_channels, kernel_size=1),
        )

    def forward(self, dense_features: torch.Tensor) -> torch.Tensor:
        if dense_features.ndim != 4:
            raise ValueError(f"Expected shape (batch, {IN_DIM}, H, W), got {tuple(dense_features.shape)}")
        projected = self.mlp(dense_features)
        return F.normalize(projected, dim=1)  # normalize each spatial cell's channel vector


if __name__ == "__main__":
    torch.manual_seed(0)

    global_head = GlobalHead()
    dense_head = DenseHead()

    pooled = torch.randn(4, IN_DIM)
    grid = torch.randn(4, IN_DIM, 32, 32)

    global_out = global_head(pooled)
    dense_out = dense_head(grid)

    print(f"Global head: {tuple(pooled.shape)} -> {tuple(global_out.shape)}")
    print(f"  L2 norm per row (expected 1.0): {global_out.norm(dim=1)[:4].tolist()}")

    print(f"Dense head:  {tuple(grid.shape)} -> {tuple(dense_out.shape)}")
    cell_norms = dense_out.norm(dim=1)  # (batch, H, W)
    print(f"  L2 norm at cell (0,0,0) and (0,16,16) (expected ~1.0): "
          f"{cell_norms[0, 0, 0].item():.4f}, {cell_norms[0, 16, 16].item():.4f}")

    # End-to-end with the real encoder + momentum twins.
    from encoder import Encoder
    from momentum import EMAModule, MomentumEncoder

    online_encoder = Encoder()
    momentum_encoder = MomentumEncoder(online_encoder)
    momentum_global_head = EMAModule(global_head)
    momentum_dense_head = EMAModule(dense_head)

    dummy = torch.zeros(2, 1, 256, 256)
    online_pooled = online_encoder(dummy, pool=True)
    online_dense = online_encoder(dummy, pool=False)
    momentum_pooled = momentum_encoder(dummy, pool=True)
    momentum_dense = momentum_encoder(dummy, pool=False)

    z_a_global = global_head(online_pooled)
    z_b_global = momentum_global_head(momentum_pooled)
    z_a_dense = dense_head(online_dense)
    z_b_dense = momentum_dense_head(momentum_dense)

    print("\nFull online + momentum path:")
    print(f"  z_a_global: {tuple(z_a_global.shape)}, requires_grad={z_a_global.requires_grad}")
    print(f"  z_b_global: {tuple(z_b_global.shape)}, requires_grad={z_b_global.requires_grad}")
    print(f"  z_a_dense:  {tuple(z_a_dense.shape)}, requires_grad={z_a_dense.requires_grad}")
    print(f"  z_b_dense:  {tuple(z_b_dense.shape)}, requires_grad={z_b_dense.requires_grad}")


In [ ]:
%%writefile /kaggle/working/densecl_src/memory_queue.py
"""FIFO memory queue of global feature vectors, MoCo-style.

The global InfoNCE loss (see `losses.py`) needs a large, diverse pool of
"different image" negatives to contrast each positive pair against, without
having to re-encode thousands of images every training step. This queue
holds a running buffer of recent momentum-encoder global vectors (`z_b`
from `heads.GlobalHead`, already L2-normalized) and reuses them as
negatives for many steps after they were computed.

Usage order matters: call `get()` to fetch the current negatives and
compute the loss BEFORE calling `enqueue()` with the current batch's
momentum vectors. Enqueueing first would let the current batch's own
positive keys leak into its own negative pool, defeating the point of using
*past* batches for diversity.

NOTE ON THE FILE NAME: deliberately not `queue.py` - that would shadow
Python's own standard-library `queue` module, which `torch`'s multi-worker
DataLoader relies on internally (`num_workers > 0` spawns worker processes
that import it). A flat-import module named `queue.py` sitting on `sys.path`
ahead of the standard library would silently break that.
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

from heads import PROJECTION_DIM

DEFAULT_QUEUE_SIZE = 4096  # ~18% of the ~22,680-image pretraining set - tunable, not load-bearing


class MemoryQueue(nn.Module):
    """Fixed-size FIFO buffer of L2-normalized global vectors."""

    def __init__(self, dim: int = PROJECTION_DIM, size: int = DEFAULT_QUEUE_SIZE) -> None:
        super().__init__()
        self.dim = dim
        self.size = size
        # Random-but-normalized init: never queried before the first enqueue()
        # in practice (the training loop should warm up briefly), but a
        # normalized init keeps early cosine similarities well-behaved if it
        # ever is.
        self.register_buffer("vectors", F.normalize(torch.randn(size, dim), dim=1))
        self.register_buffer("pointer", torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def enqueue(self, keys: torch.Tensor) -> None:
        """Push a batch of L2-normalized momentum-encoder global vectors in,
        overwriting the oldest entries (circular buffer)."""
        batch_size = keys.shape[0]
        if batch_size > self.size:
            keys = keys[-self.size:]
            batch_size = keys.shape[0]

        ptr = int(self.pointer.item())
        end = ptr + batch_size
        if end <= self.size:
            self.vectors[ptr:end] = keys
        else:
            first_chunk = self.size - ptr
            self.vectors[ptr:] = keys[:first_chunk]
            self.vectors[: end - self.size] = keys[first_chunk:]
        self.pointer[0] = end % self.size

    def get(self) -> torch.Tensor:
        """Current queue contents, `(size, dim)`. Cloned defensively so a
        later `enqueue()` can't mutate a tensor a caller is still using."""
        return self.vectors.clone()


if __name__ == "__main__":
    queue = MemoryQueue(dim=8, size=10)
    print(f"Initial queue shape: {tuple(queue.get().shape)}")

    batch1 = F.normalize(torch.ones(4, 8), dim=1)
    queue.enqueue(batch1)
    print(f"After enqueueing 4: pointer={int(queue.pointer.item())}")
    print(f"Rows 0-3 match batch1: {torch.allclose(queue.get()[:4], batch1)}")

    batch2 = F.normalize(torch.full((8, 8), 2.0), dim=1)
    queue.enqueue(batch2)  # 4 + 8 = 12 > size 10 -> must wrap around
    print(f"After enqueueing 8 more (size=10, wraps around): pointer={int(queue.pointer.item())}")
    contents = queue.get()
    print(f"Rows 4-9 hold the first 6 of batch2: {torch.allclose(contents[4:10], batch2[:6])}")
    print(f"Rows 0-1 hold the wrapped remainder of batch2: {torch.allclose(contents[0:2], batch2[6:8])}")


In [ ]:
%%writefile /kaggle/working/densecl_src/losses.py
"""Contrastive losses for DenseCL pretraining.

Two losses, combined: global InfoNCE (whole-image instance discrimination)
and dense foreground-masked correspondence (patch-level instance
discrimination). See `progress_so_far.md` Section 2 for how these plug into
the rest of the architecture, and the earlier chat discussion of "what is
the loss function" / "what is the patch matching technique" for the
reasoning behind the design choices below - this module implements exactly
that discussion, not a different design.
"""

from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class LossConfig:
    """Tunable loss hyperparameters. Defaults are standard MoCo/DenseCL
    starting points, not final values - meant to be swept against
    downstream validation AUC later (lambda_dense especially - see
    `progress_so_far.md` Section 5, "lambda_dense sweep"), the same way
    `AugmentConfig` and `min_coverage` are treated."""

    temperature_global: float = 0.2
    temperature_dense: float = 0.2
    lambda_global: float = 1.0
    lambda_dense: float = 1.0


DEFAULT_LOSS_CONFIG = LossConfig()


class GlobalInfoNCELoss(nn.Module):
    """Whole-image instance discrimination: pull View A's and View B's
    global vectors together, push apart from every vector in the memory
    queue (see `memory_queue.py`).

    Standard MoCo formulation: a `(1 + queue_size)`-way classification
    problem where the true positive pair is always placed at index 0, and
    cross-entropy is used to score how confidently the model picks it out
    against every queue negative.
    """

    def __init__(self, temperature: float = DEFAULT_LOSS_CONFIG.temperature_global) -> None:
        super().__init__()
        self.temperature = temperature

    def forward(self, z_a: torch.Tensor, z_b: torch.Tensor, queue: torch.Tensor) -> torch.Tensor:
        """z_a: (batch, dim) online View A vectors (query, gradients flow).
        z_b: (batch, dim) momentum View B vectors (positive key, no grad).
        queue: (queue_size, dim) momentum vectors from past batches
        (negative keys, no grad). All rows are assumed L2-normalized
        already (both `GlobalHead` and `MemoryQueue` guarantee this), so
        dot products here are cosine similarities."""
        pos_logits = (z_a * z_b).sum(dim=1, keepdim=True) / self.temperature       # (batch, 1)
        neg_logits = z_a @ queue.T / self.temperature                               # (batch, queue_size)
        logits = torch.cat([pos_logits, neg_logits], dim=1)                         # (batch, 1+queue_size)
        labels = torch.zeros(z_a.shape[0], dtype=torch.long, device=z_a.device)     # positive is always index 0
        return F.cross_entropy(logits, labels)


class DenseCorrespondenceLoss(nn.Module):
    """Patch-level instance discrimination, restricted to ink-bearing
    (foreground) grid cells only.

    For each foreground patch in View A, find its nearest neighbor (highest
    cosine similarity) among View B's foreground patches OF THE SAME
    signature - that becomes the positive pair (matching never crosses
    images: two different signatures' patches have no correspondence to
    find in the first place). Negatives are every foreground patch in View
    B across the WHOLE batch.

    Deliberate simplification vs. the original DenseCL paper: negatives
    come from an in-batch pool, not a separate dense memory queue. A dense
    queue would need one slot per ink-bearing grid cell rather than one per
    image - tens of thousands of slots instead of a few thousand - which is
    real added complexity for a first working version. Upgradeable later if
    the in-batch negative pool proves too small in practice.

    Matching runs per-sample in a plain Python loop over the batch rather
    than a single fully-vectorized op - clearer to read and get right, and
    batch sizes here are small enough (tens, not thousands) that this is
    not a bottleneck.
    """

    def __init__(self, temperature: float = DEFAULT_LOSS_CONFIG.temperature_dense) -> None:
        super().__init__()
        self.temperature = temperature

    def forward(
        self,
        z_a: torch.Tensor,
        z_b: torch.Tensor,
        mask_a: torch.Tensor,
        mask_b: torch.Tensor,
    ) -> torch.Tensor:
        """z_a/z_b: (batch, dim, H, W) online/momentum dense projected
        features (L2-normalized per cell, from `DenseHead`). mask_a/mask_b:
        (batch, H, W) bool foreground masks at the same H, W (from
        `mask.py`'s `compute_foreground_mask`, computed per-view - see the
        earlier chat explanation of why masking must be recomputed per
        augmented view rather than reused from the original image)."""
        batch_size, dim, h, w = z_a.shape

        feat_a = z_a.permute(0, 2, 3, 1).reshape(batch_size, h * w, dim)
        feat_b = z_b.permute(0, 2, 3, 1).reshape(batch_size, h * w, dim)
        flat_mask_a = mask_a.reshape(batch_size, h * w)
        flat_mask_b = mask_b.reshape(batch_size, h * w)

        # Negative pool: every foreground patch in View B, across the whole batch.
        all_keys = feat_b[flat_mask_b]  # (total_fg_in_batch, dim)
        if all_keys.shape[0] == 0:
            return z_a.new_zeros(())  # degenerate: no ink anywhere in the batch's View B

        per_sample_losses = []
        for n in range(batch_size):
            query_idx = flat_mask_a[n].nonzero(as_tuple=True)[0]
            key_idx = flat_mask_b[n].nonzero(as_tuple=True)[0]
            if query_idx.numel() == 0 or key_idx.numel() == 0:
                continue  # this view has no ink (shouldn't normally happen post-preprocessing)

            queries = feat_a[n, query_idx]           # (Nq, dim)
            keys_same_sample = feat_b[n, key_idx]    # (Nk, dim)

            # Nearest-neighbor matching is a hard index selection with no
            # gradient of its own - compute it under no_grad so we don't
            # build an autograd graph for a similarity matrix we only ever
            # call argmax() on.
            with torch.no_grad():
                match_similarity = queries @ keys_same_sample.T   # (Nq, Nk)
                best_match = match_similarity.argmax(dim=1)        # (Nq,)
            positives = keys_same_sample[best_match]                # (Nq, dim)

            pos_logits = (queries * positives).sum(dim=1, keepdim=True) / self.temperature
            neg_logits = queries @ all_keys.T / self.temperature
            logits = torch.cat([pos_logits, neg_logits], dim=1)
            labels = torch.zeros(queries.shape[0], dtype=torch.long, device=queries.device)
            per_sample_losses.append(F.cross_entropy(logits, labels))

        if not per_sample_losses:
            return z_a.new_zeros(())
        return torch.stack(per_sample_losses).mean()


class DenseCLLoss(nn.Module):
    """Combines both losses: total = lambda_global * L_global + lambda_dense * L_dense."""

    def __init__(self, config: LossConfig = DEFAULT_LOSS_CONFIG) -> None:
        super().__init__()
        self.config = config
        self.global_loss = GlobalInfoNCELoss(temperature=config.temperature_global)
        self.dense_loss = DenseCorrespondenceLoss(temperature=config.temperature_dense)

    def forward(
        self,
        z_a_global: torch.Tensor,
        z_b_global: torch.Tensor,
        queue: torch.Tensor,
        z_a_dense: torch.Tensor,
        z_b_dense: torch.Tensor,
        mask_a: torch.Tensor,
        mask_b: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        loss_global = self.global_loss(z_a_global, z_b_global, queue)
        loss_dense = self.dense_loss(z_a_dense, z_b_dense, mask_a, mask_b)
        total = self.config.lambda_global * loss_global + self.config.lambda_dense * loss_dense
        return {"total": total, "global": loss_global, "dense": loss_dense}


if __name__ == "__main__":
    torch.manual_seed(0)

    # ---- Global loss: behavioral sanity check (not just shapes) ----
    # A loss that "works" should score a true positive pair as clearly more
    # likely than an unrelated pair, given the same random negatives.
    batch_size, dim, queue_size = 4, 128, 100
    global_loss_fn = GlobalInfoNCELoss(temperature=0.2)
    queue_vectors = F.normalize(torch.randn(queue_size, dim), dim=1)

    z_a = F.normalize(torch.randn(batch_size, dim), dim=1)
    z_b_matched = z_a.clone()
    z_b_random = F.normalize(torch.randn(batch_size, dim), dim=1)

    loss_matched = global_loss_fn(z_a, z_b_matched, queue_vectors)
    loss_random = global_loss_fn(z_a, z_b_random, queue_vectors)
    print(f"Global loss (true positive pair):  {loss_matched.item():.4f} (expect low)")
    print(f"Global loss (unrelated pair):       {loss_random.item():.4f} (expect higher)")
    assert loss_matched.item() < loss_random.item(), "Global loss must reward true positives over random pairs"

    # ---- Dense loss: same kind of behavioral check ----
    grid = 32
    dense_loss_fn = DenseCorrespondenceLoss(temperature=0.2)
    z_a_dense = F.normalize(torch.randn(batch_size, dim, grid, grid), dim=1)
    z_b_dense_matched = z_a_dense.clone()
    z_b_dense_random = F.normalize(torch.randn(batch_size, dim, grid, grid), dim=1)

    mask_a = torch.rand(batch_size, grid, grid) < 0.15  # ~matches the ~12% ink density measured on real data
    mask_b = torch.rand(batch_size, grid, grid) < 0.15
    mask_a[:, 0, 0] = True  # guarantee non-degenerate (at least one fg cell) in this synthetic test
    mask_b[:, 0, 0] = True

    loss_dense_matched = dense_loss_fn(z_a_dense, z_b_dense_matched, mask_a, mask_a)
    loss_dense_random = dense_loss_fn(z_a_dense, z_b_dense_random, mask_a, mask_b)
    print(f"Dense loss (identical view, same mask): {loss_dense_matched.item():.4f} (expect low)")
    print(f"Dense loss (unrelated views/masks):     {loss_dense_random.item():.4f} (expect higher)")
    assert loss_dense_matched.item() < loss_dense_random.item(), "Dense loss must reward true correspondences"

    # ---- Full pipeline, real data, real backward pass ----
    import sys
    from pathlib import Path

    sys.path.insert(0, str(Path(__file__).resolve().parent))
    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "utils"))
    from torch.utils.data import DataLoader

    from dataset import SignatureSSLDataset
    from encoder import Encoder
    from heads import DenseHead, GlobalHead
    from memory_queue import MemoryQueue
    from momentum import EMAModule, MomentumEncoder

    DATA_ROOT = Path(__file__).resolve().parents[2] / "data" / "all"
    ds = SignatureSSLDataset(DATA_ROOT)
    loader = DataLoader(ds, batch_size=8, shuffle=True, num_workers=0)
    real_batch = next(iter(loader))

    online_encoder = Encoder()
    momentum_encoder = MomentumEncoder(online_encoder)
    global_head = GlobalHead()
    dense_head = DenseHead()
    momentum_global_head = EMAModule(global_head)
    momentum_dense_head = EMAModule(dense_head)
    memory_queue = MemoryQueue()
    dense_cl_loss = DenseCLLoss()

    z_a_global = global_head(online_encoder(real_batch["view_a"], pool=True))
    z_a_dense = dense_head(online_encoder(real_batch["view_a"], pool=False))
    z_b_global = momentum_global_head(momentum_encoder(real_batch["view_b"], pool=True))
    z_b_dense = momentum_dense_head(momentum_encoder(real_batch["view_b"], pool=False))

    losses = dense_cl_loss(
        z_a_global, z_b_global, memory_queue.get(),
        z_a_dense, z_b_dense, real_batch["mask_a"], real_batch["mask_b"],
    )
    print(f"\nReal-data losses: total={losses['total'].item():.4f}, "
          f"global={losses['global'].item():.4f}, dense={losses['dense'].item():.4f}")

    losses["total"].backward()
    grad = online_encoder.stem.layers[0].block[0].weight.grad
    print(f"backward() ok, gradient reached the encoder's first layer: {grad is not None and grad.abs().sum().item() > 0}")

    # Only AFTER computing the loss: enqueue this batch's momentum keys for future steps.
    memory_queue.enqueue(z_b_global.detach())
    print(f"Queue pointer after one enqueue: {int(memory_queue.pointer.item())}")


In [ ]:
%%writefile /kaggle/working/densecl_src/lr_scheduler.py
"""Linear-warmup, cosine-annealed learning rate schedule.

Ported from the original thesis's
`Thesis_Final/ssl_pretraining/learningratescheduler/cosineLR.py` (the same
schedule already used to train the existing reconstruction-SSL baseline),
reusing a validated implementation rather than writing a new one from
scratch. Only the base class import was updated (`LRScheduler`, the current
public name - the original's `_LRScheduler` still works in this venv's
torch 2.8.0 but is the deprecated private alias).
"""

from __future__ import annotations

import math
import warnings

from torch.optim.lr_scheduler import LRScheduler


class LinearWarmupCosineAnnealingLR(LRScheduler):
    def __init__(
        self,
        optimizer,
        warmup_steps: int,
        total_steps: int,
        warmup_start_lr: float = 0.0,
        eta_min: float = 0.0,
        last_epoch: int = -1,
    ) -> None:
        self.warmup_epochs = int(warmup_steps)
        self.max_epochs = int(total_steps)
        self.warmup_start_lr = warmup_start_lr
        self.eta_min = eta_min
        super().__init__(optimizer, last_epoch)

    def get_lr(self) -> list[float]:
        if not self._get_lr_called_within_step:
            warnings.warn(
                "To get the last learning rate computed by the scheduler, please use `get_last_lr()`.",
                UserWarning,
            )

        if self.last_epoch == 0:
            return [self.warmup_start_lr] * len(self.base_lrs)
        elif self.last_epoch < self.warmup_epochs:
            return [
                group["lr"] + (base_lr - self.warmup_start_lr) / (self.warmup_epochs - 1)
                for base_lr, group in zip(self.base_lrs, self.optimizer.param_groups)
            ]
        elif self.last_epoch == self.warmup_epochs:
            return self.base_lrs
        elif (self.last_epoch - 1 - self.max_epochs) % (2 * (self.max_epochs - self.warmup_epochs)) == 0:
            return [
                group["lr"] + (base_lr - self.eta_min) *
                (1 - math.cos(math.pi / (self.max_epochs - self.warmup_epochs))) / 2
                for base_lr, group in zip(self.base_lrs, self.optimizer.param_groups)
            ]

        return [
            (1 + math.cos(math.pi * (self.last_epoch - self.warmup_epochs) / (self.max_epochs - self.warmup_epochs))) /
            (
                1 +
                math.cos(math.pi * (self.last_epoch - self.warmup_epochs - 1) / (self.max_epochs - self.warmup_epochs))
            ) * (group["lr"] - self.eta_min) + self.eta_min for group in self.optimizer.param_groups
        ]

    def _get_closed_form_lr(self) -> list[float]:
        if self.last_epoch < self.warmup_epochs:
            return [
                self.warmup_start_lr + self.last_epoch * (base_lr - self.warmup_start_lr) / (self.warmup_epochs - 1)
                for base_lr in self.base_lrs
            ]

        return [
            self.eta_min + 0.5 * (base_lr - self.eta_min) *
            (1 + math.cos(math.pi * (self.last_epoch - self.warmup_epochs) / (self.max_epochs - self.warmup_epochs)))
            for base_lr in self.base_lrs
        ]


In [ ]:
%%writefile /kaggle/working/densecl_src/train.py
"""DenseCL pretraining loop - Kaggle version.

Identical logic to the local project's `driver/train.py`, wiring together
`dataset.py`, `encoder.py`, `heads.py`, `momentum.py`, `memory_queue.py`,
`losses.py` into an actual training run. The only changes from the local
version: `DATA_ROOT`/`RESULTS_DIR` come from `kaggle_config.py` (the
notebook's single editable config cell) instead of being derived from this
file's own location on disk; the `sys.path` bootstrapping is gone since the
Kaggle notebook keeps every module flat in one directory, already on
`sys.path`; and there is no `if __name__ == "__main__": train()` auto-run at
the bottom - the notebook's own final cell builds a `TrainConfig` explicitly
from `kaggle_config.py`'s values and calls `train(config)` itself, so every
knob is visibly wired from that one config cell.

One training step, in order (the order matters - see the module-level
comment in `train_one_step`):
  1. pull a batch (view_a, view_b, mask_a, mask_b)
  2. forward View A through the online encoder + online heads (grad on)
  3. forward View B through the momentum encoder + momentum heads (no grad)
  4. fetch the queue's CURRENT contents (before this step touches it)
  5. compute DenseCLLoss
  6. backward + optimizer step (updates only the online encoder/heads)
  7. EMA-update the momentum encoder/heads toward the just-updated online weights
  8. enqueue this batch's momentum global vectors for FUTURE steps' negatives
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from torch.optim import SGD
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from kaggle_config import DATA_ROOT, RESULTS_DIR
from dataset import SignatureSSLDataset, list_specific_writer_signature_paths
from encoder import Encoder
from heads import DenseHead, GlobalHead
from losses import DenseCLLoss, LossConfig
from lr_scheduler import LinearWarmupCosineAnnealingLR
from memory_queue import DEFAULT_QUEUE_SIZE, MemoryQueue
from momentum import DEFAULT_MOMENTUM, EMAModule, MomentumEncoder
from validation_set_creation import VALIDATION_WRITER_COUNTS, load_validation_writer_ids


@dataclass(frozen=True)
class TrainConfig:
    """Defaults here are just a fallback - the notebook's final cell always
    builds this explicitly from `kaggle_config.py`'s values, so editing
    that one cell is enough; these defaults only matter if `TrainConfig()`
    is ever constructed with no arguments."""

    run_name: str = "densecl_pretrain_v1"

    num_epochs: int = 5
    warmup_epochs: int = 1
    save_frequency: int = 5  # epochs

    batch_size: int = 8
    num_workers: int = 2

    base_lr: float = 0.03
    base_lr_batch_size: int = 256
    sgd_momentum: float = 0.9
    weight_decay: float = 1e-4

    encoder_momentum: float = DEFAULT_MOMENTUM  # EMA momentum for the momentum encoder/heads
    queue_size: int = DEFAULT_QUEUE_SIZE
    loss_config: LossConfig = LossConfig()

    seed: int = 42

    @property
    def learning_rate(self) -> float:
        return self.base_lr * self.batch_size / self.base_lr_batch_size


@dataclass
class Models:
    online_encoder: Encoder
    momentum_encoder: MomentumEncoder
    global_head: GlobalHead
    dense_head: DenseHead
    momentum_global_head: EMAModule
    momentum_dense_head: EMAModule
    memory_queue: MemoryQueue
    loss_fn: DenseCLLoss

    def to(self, device: torch.device) -> "Models":
        for module in (
            self.online_encoder, self.momentum_encoder, self.global_head, self.dense_head,
            self.momentum_global_head, self.momentum_dense_head, self.memory_queue, self.loss_fn,
        ):
            module.to(device)
        return self

    def train(self) -> None:
        self.online_encoder.train()
        self.global_head.train()
        self.dense_head.train()
        # momentum modules stay in eval-equivalent behavior via their own
        # torch.no_grad() forward passes regardless of .train()/.eval(),
        # but BatchNorm still needs an explicit mode - keep them in train()
        # mode so their running stats keep updating from the momentum path's
        # own inputs too (matches standard MoCo practice).
        self.momentum_encoder.train()
        self.momentum_global_head.train()
        self.momentum_dense_head.train()

    def eval(self) -> None:
        self.online_encoder.eval()
        self.global_head.eval()
        self.dense_head.eval()
        self.momentum_encoder.eval()
        self.momentum_global_head.eval()
        self.momentum_dense_head.eval()

    def trainable_parameters(self):
        return list(self.online_encoder.parameters()) + list(self.global_head.parameters()) + list(self.dense_head.parameters())


def build_models(config: TrainConfig) -> Models:
    online_encoder = Encoder()
    momentum_encoder = MomentumEncoder(online_encoder)
    global_head = GlobalHead()
    dense_head = DenseHead()
    momentum_global_head = EMAModule(global_head)
    momentum_dense_head = EMAModule(dense_head)
    memory_queue = MemoryQueue(size=config.queue_size)
    loss_fn = DenseCLLoss(config=config.loss_config)
    return Models(
        online_encoder, momentum_encoder, global_head, dense_head,
        momentum_global_head, momentum_dense_head, memory_queue, loss_fn,
    )


def build_dataloaders(config: TrainConfig) -> tuple[DataLoader, DataLoader]:
    """Writer-level train/validation split: validation writers come from
    `validation_set_creation.py`'s fixed, pre-created split (already
    guaranteed disjoint from the test split by construction); the training
    pool is every remaining writer after excluding BOTH test and validation
    writers. No images are kept around for a separate "sanity" mechanism -
    see `evaluate_validation_loss`."""
    validation_writer_ids = {
        dataset_name: load_validation_writer_ids(dataset_name)
        for dataset_name in VALIDATION_WRITER_COUNTS
    }

    train_dataset = SignatureSSLDataset(DATA_ROOT, extra_exclude_writer_ids=validation_writer_ids)

    validation_paths = list_specific_writer_signature_paths(DATA_ROOT, validation_writer_ids)
    validation_dataset = SignatureSSLDataset(DATA_ROOT, image_paths_override=validation_paths)

    # Note: SignatureSSLDataset.__getitem__ draws a fresh, UNSEEDED
    # np.random.default_rng() per call, by design (see dataset.py's
    # docstring - re-augments every epoch, View A != View B). That's left
    # untouched here; only the DataLoader's shuffling order is made
    # reproducible via `loader_generator`.
    loader_generator = torch.Generator().manual_seed(config.seed)
    train_loader = DataLoader(
        train_dataset, batch_size=config.batch_size, shuffle=True,
        num_workers=config.num_workers, generator=loader_generator, drop_last=True,
    )
    validation_loader = DataLoader(
        validation_dataset, batch_size=config.batch_size, shuffle=False,
        num_workers=config.num_workers,
    )
    return train_loader, validation_loader


def train_one_step(
    batch: dict[str, torch.Tensor],
    models: Models,
    optimizer: SGD,
    scheduler: LinearWarmupCosineAnnealingLR,
    config: TrainConfig,
    device: torch.device,
) -> dict[str, float]:
    view_a = batch["view_a"].to(device, non_blocking=True)
    view_b = batch["view_b"].to(device, non_blocking=True)
    mask_a = batch["mask_a"].to(device, non_blocking=True)
    mask_b = batch["mask_b"].to(device, non_blocking=True)

    optimizer.zero_grad(set_to_none=True)

    # Step 2: online path (View A), gradients on.
    pooled_a = models.online_encoder(view_a, pool=True)
    dense_a = models.online_encoder(view_a, pool=False)
    z_a_global = models.global_head(pooled_a)
    z_a_dense = models.dense_head(dense_a)

    # Step 3: momentum path (View B), no gradients (enforced inside MomentumEncoder/EMAModule).
    pooled_b = models.momentum_encoder(view_b, pool=True)
    dense_b = models.momentum_encoder(view_b, pool=False)
    z_b_global = models.momentum_global_head(pooled_b)
    z_b_dense = models.momentum_dense_head(dense_b)

    # Step 4: read the queue's CURRENT contents, before this step's enqueue().
    queue_vectors = models.memory_queue.get()

    # Step 5: loss.
    losses = models.loss_fn(z_a_global, z_b_global, queue_vectors, z_a_dense, z_b_dense, mask_a, mask_b)

    # Step 6: backward + optimizer step - updates only the online encoder/heads.
    losses["total"].backward()
    optimizer.step()
    scheduler.step()

    # Step 7: EMA-update the momentum encoder/heads toward the just-updated online weights.
    models.momentum_encoder.update(models.online_encoder, momentum=config.encoder_momentum)
    models.momentum_global_head.update(models.global_head, momentum=config.encoder_momentum)
    models.momentum_dense_head.update(models.dense_head, momentum=config.encoder_momentum)

    # Step 8: only now enqueue this batch's momentum keys, for future steps' negatives.
    models.memory_queue.enqueue(z_b_global.detach())

    return {key: float(value.item()) for key, value in losses.items()}


@torch.no_grad()
def evaluate_validation_loss(validation_loader: DataLoader, models: Models, device: torch.device) -> dict[str, float]:
    """Forward-only loss on held-out WRITERS the training loop never trains
    on - the classical overfitting check, tracked as numbers only (no images
    kept). Never touches the queue's contents (no enqueue) - validation
    batches must not contaminate the negative pool training steps rely on."""
    models.eval()
    totals, globals_, denses = [], [], []

    progress_bar = tqdm(validation_loader, desc="  Validation", leave=False)
    for batch in progress_bar:
        view_a = batch["view_a"].to(device, non_blocking=True)
        view_b = batch["view_b"].to(device, non_blocking=True)
        mask_a = batch["mask_a"].to(device, non_blocking=True)
        mask_b = batch["mask_b"].to(device, non_blocking=True)

        z_a_global = models.global_head(models.online_encoder(view_a, pool=True))
        z_a_dense = models.dense_head(models.online_encoder(view_a, pool=False))
        z_b_global = models.momentum_global_head(models.momentum_encoder(view_b, pool=True))
        z_b_dense = models.momentum_dense_head(models.momentum_encoder(view_b, pool=False))

        losses = models.loss_fn(
            z_a_global, z_b_global, models.memory_queue.get(),
            z_a_dense, z_b_dense, mask_a, mask_b,
        )
        totals.append(losses["total"].item())
        globals_.append(losses["global"].item())
        denses.append(losses["dense"].item())
        progress_bar.set_postfix({
            "total": f"{sum(totals) / len(totals):.4f}",
            "global": f"{sum(globals_) / len(globals_):.4f}",
            "dense": f"{sum(denses) / len(denses):.4f}",
        })

    models.train()
    return {
        "validation_total": sum(totals) / len(totals),
        "validation_global": sum(globals_) / len(globals_),
        "validation_dense": sum(denses) / len(denses),
    }


def save_checkpoint(models: Models, optimizer: SGD, scheduler: LinearWarmupCosineAnnealingLR,
                     epoch: int, global_step: int, config: TrainConfig, run_dir: Path) -> None:
    checkpoint_dir = run_dir / "checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    # Full state, for resuming a run.
    torch.save({
        "epoch": epoch,
        "global_step": global_step,
        "online_encoder": models.online_encoder.state_dict(),
        "momentum_encoder": models.momentum_encoder.state_dict(),
        "global_head": models.global_head.state_dict(),
        "dense_head": models.dense_head.state_dict(),
        "momentum_global_head": models.momentum_global_head.state_dict(),
        "momentum_dense_head": models.momentum_dense_head.state_dict(),
        "memory_queue": models.memory_queue.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
    }, checkpoint_dir / f"train_state_epoch{epoch}.pt")

    # Encoder-only, the actual artifact Stage B (downstream) reuses.
    torch.save(models.online_encoder.state_dict(), checkpoint_dir / f"encoder_epoch{epoch}.pt")

    print(f"  [Checkpoint] Saved train_state_epoch{epoch}.pt and encoder_epoch{epoch}.pt")


def train(config: TrainConfig = TrainConfig()) -> None:
    torch.manual_seed(config.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(config.seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader, validation_loader = build_dataloaders(config)
    models = build_models(config).to(device)
    models.train()

    optimizer = SGD(
        models.trainable_parameters(),
        lr=config.learning_rate,
        momentum=config.sgd_momentum,
        weight_decay=config.weight_decay,
    )

    steps_per_epoch = len(train_loader)
    warmup_steps = max(1, config.warmup_epochs * steps_per_epoch)
    total_steps = max(warmup_steps + 1, config.num_epochs * steps_per_epoch)
    scheduler = LinearWarmupCosineAnnealingLR(optimizer, warmup_steps=warmup_steps, total_steps=total_steps)

    run_dir = RESULTS_DIR / config.run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    step_csv = run_dir / "step_summary.csv"
    epoch_csv = run_dir / "epoch_summary.csv"

    print(f"Run             : {config.run_name}")
    print(f"Device          : {device}")
    print(f"Dataset         : {len(train_loader.dataset)} train / {len(validation_loader.dataset)} validation (writer-level, fixed split)")
    print(f"Epochs          : {config.num_epochs}")
    print(f"Steps per epoch : {steps_per_epoch}")
    print(f"Warmup steps    : {warmup_steps}")
    print(f"Total steps     : {total_steps}")
    print(f"Batch size      : {config.batch_size}")
    print(f"Learning rate   : {config.learning_rate:.5f} (base {config.base_lr} scaled for batch {config.batch_size})")
    print(f"Queue size      : {config.queue_size}")
    print(f"Encoder momentum: {config.encoder_momentum}")
    print("=" * 60)

    step_logs: list[dict] = []
    epoch_logs: list[dict] = []
    global_step = 0

    for epoch in range(1, config.num_epochs + 1):
        epoch_totals: list[float] = []
        epoch_globals: list[float] = []
        epoch_denses: list[float] = []

        progress_bar = tqdm(train_loader, desc=f"[{config.run_name}] Epoch {epoch}/{config.num_epochs}", leave=True)
        for batch_index, batch in enumerate(progress_bar):
            losses = train_one_step(batch, models, optimizer, scheduler, config, device)
            global_step += 1

            epoch_totals.append(losses["total"])
            epoch_globals.append(losses["global"])
            epoch_denses.append(losses["dense"])

            step_logs.append({
                "run_name": config.run_name,
                "epoch": epoch,
                "epoch_step": batch_index + 1,
                "global_step": global_step,
                "loss_total": losses["total"],
                "loss_global": losses["global"],
                "loss_dense": losses["dense"],
                "lr": optimizer.param_groups[0]["lr"],
            })
            progress_bar.set_postfix({
                "total": f"{losses['total']:.4f}",
                "global": f"{losses['global']:.4f}",
                "dense": f"{losses['dense']:.4f}",
                "lr": f"{optimizer.param_groups[0]['lr']:.5f}",
            })

        validation_metrics = evaluate_validation_loss(validation_loader, models, device)
        epoch_log = {
            "run_name": config.run_name,
            "epoch": epoch,
            "epoch_avg_loss_total": sum(epoch_totals) / len(epoch_totals),
            "epoch_avg_loss_global": sum(epoch_globals) / len(epoch_globals),
            "epoch_avg_loss_dense": sum(epoch_denses) / len(epoch_denses),
            **validation_metrics,
        }
        epoch_logs.append(epoch_log)
        print(f"  [Epoch {epoch}] train_total={epoch_log['epoch_avg_loss_total']:.4f}  "
              f"validation_total={validation_metrics['validation_total']:.4f}")

        pd.DataFrame(step_logs).to_csv(step_csv, index=False)
        pd.DataFrame(epoch_logs).to_csv(epoch_csv, index=False)

        if epoch % config.save_frequency == 0 or epoch == config.num_epochs:
            save_checkpoint(models, optimizer, scheduler, epoch, global_step, config, run_dir)


## 5. Run training

In [ ]:
from kaggle_config import (
    BASE_LR, BASE_LR_BATCH_SIZE, BATCH_SIZE, ENCODER_MOMENTUM, LAMBDA_DENSE,
    LAMBDA_GLOBAL, NUM_EPOCHS, NUM_WORKERS, QUEUE_SIZE, RUN_NAME,
    SAVE_FREQUENCY, SEED, SGD_MOMENTUM, TEMPERATURE_DENSE, TEMPERATURE_GLOBAL,
    WARMUP_EPOCHS, WEIGHT_DECAY,
)
from losses import LossConfig
from train import TrainConfig, train

config = TrainConfig(
    run_name=RUN_NAME,
    num_epochs=NUM_EPOCHS,
    warmup_epochs=WARMUP_EPOCHS,
    save_frequency=SAVE_FREQUENCY,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    base_lr=BASE_LR,
    base_lr_batch_size=BASE_LR_BATCH_SIZE,
    sgd_momentum=SGD_MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    encoder_momentum=ENCODER_MOMENTUM,
    queue_size=QUEUE_SIZE,
    loss_config=LossConfig(
        temperature_global=TEMPERATURE_GLOBAL,
        temperature_dense=TEMPERATURE_DENSE,
        lambda_global=LAMBDA_GLOBAL,
        lambda_dense=LAMBDA_DENSE,
    ),
    seed=SEED,
)

train(config)


## 6. Verify results

In [ ]:
import pandas as pd

from kaggle_config import RESULTS_DIR, RUN_NAME

run_dir = RESULTS_DIR / RUN_NAME
epoch_summary = pd.read_csv(run_dir / "epoch_summary.csv")
print(epoch_summary.to_string(index=False))

checkpoint_dir = run_dir / "checkpoints"
print("\nCheckpoints saved:")
for path in sorted(checkpoint_dir.glob("*.pt")):
    print(f"  {path.name}  ({path.stat().st_size / 1e6:.1f} MB)")
